**Fig. 6 and 7, step 4 (notebook 04): draw the niche panels, among them Fig. 6C, 6H and 7A-D**

In [ ]:
# Working directory: data/fig6_7/ under the repository root, where the Fig. 6 and 7 deposit folder is downloaded
# (all_stacks_features.csv, workspace/ = the plugin's Step 4 workspace with the 14 stacks (13 analysed) and their meshes,
# run/ = what these notebooks write and read back, figures/ = the saved panels).
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig6_7")
print("working directory:", os.getcwd())

# Niche Figure Panels

Companion to `03_figure_panels.ipynb`. Loads the spatial-niche embedding from
`02_niche_embedding.ipynb` and produces figure-ready panels in the same visual
language as 03.

**Inputs:** `morphospace_results.pkl`, `niche_results.pkl`
**Outputs:** plotly figures + exported PNGs/CSVs under `run/`

**Structure:**
1. Setup (palettes, typography, helpers) — matches notebook 03 aesthetic
2. Data loading + niche-config selection + df merge
3. Plot functions (reusable across shape + niche embeddings)
4. One cell per visualization — run independently

Niche config is chosen at the top of the loader cell. Switching configs is a
one-line change; rerun the loader and every downstream panel refreshes.

---


## 1. Setup — palettes, typography, helpers

Copy-paste of notebook 03's setup with a single addition: a niche cluster
palette generator. Run once.


In [ ]:
import os, pickle, glob, gc, json
import numpy as np
import pandas as pd
import plotly.graph_objects as go


# -----------------------------------------------------------------------
# Typography (matches notebook 03)
# -----------------------------------------------------------------------
FONT_FAMILY       = "Helvetica Neue, Arial"
FONT_AXIS_TITLE   = 13
FONT_TICK         = 11
FONT_LEGEND       = 10.5
FONT_LEGEND_TITLE = 11


# -----------------------------------------------------------------------
# Sample ordering + short names — same STACK_ORDER as notebook 03
# -----------------------------------------------------------------------
STACK_ORDER = [
    # back (trunk skin -- its own region, not the hand)
    "(13 wks) Back skin x40-Smooth05_Resampled.tif",
    "(14 wks) back skin x40-Smooth05_Resampled.tif",
    "(12 wks) back skin x40-Smooth05_Resampled.tif",
    # front skin
    "(13 wks) Front skin x40-Smooth05_Resampled.tif",
    "(12 wks) front skin x40-Smooth05_Resampled.tif",
    # dorsal (back of the digit)
    "(13 wks) D3L Dorsal skin x40-Smooth05_Resampled.tif",
    "(14 wks) D3L dorsal x40-Smooth05_Resampled.tif",
    "(12 wks) D3L dorsal x40 stack-Smooth05_Resampled.tif",
    # volar, unpatterned
    "(14 wks) digit 5 unpatterned skin x40 stack-Smooth05_Resampled.tif",
    "(13 wks) D3L Ventral unpatterned skin x40-Smooth05_Resampled.tif",
    # volar, ridged
    "(13 wks) D3L ventral ridges x40-Smooth05_Resampled.tif",
    "(12 wks) D2L ventral tip x40-Smooth05_Resampled.tif",
    # volar, whorl
    "(14 wks) ventral whorl x40-Smooth05_Resampled.tif",
]
SHORT = {s: s.replace("-Smooth05_Resampled.tif", "").replace(" x40 stack", "").replace(" x40", "") for s in STACK_ORDER}


# Hue families: greens = back skin, magentas = front skin, teals = dorsal
# digit, yellows = volar unpatterned, oranges/reds = volar ridged,
# blue = volar whorl
SAMPLE_PALETTE = {
    "(13 wks) Back skin":                    "#117733",
    "(14 wks) back skin":                    "#66BB77",
    "(12 wks) back skin":                    "#228833",
    "(13 wks) D3L Dorsal skin":              "#44AA99",
    "(14 wks) D3L dorsal":              "#88CCAA",
    "(12 wks) D3L dorsal":              "#99DDBB",
    "(14 wks) digit 5 unpatterned skin":      "#DDCC77",
    "(13 wks) D3L Ventral unpatterned skin": "#BB9944",
    "(13 wks) D3L ventral ridges":      "#CC3311",
    "(12 wks) D2L ventral tip":         "#EE7733",
    "(14 wks) ventral whorl":           "#0077BB",
    "(13 wks) Front skin":                   "#AA3377",
    "(12 wks) front skin":                   "#EE3377",
}

PATTERN_PALETTE = {
    "back":        "#117733",   # green   -- the back-skin samples' hue family
    "front":       "#AA3377",   # magenta -- the front-skin samples' hue family
    "dorsal":      "#44AA99",   # teal    -- the dorsal-digit samples' hue family
    "unpatterned": "#DDCC77",
    "ridged":      "#CC3311",
    "whorl":       "#0077BB",
}
PATTERN_ORDER = ["back", "front", "dorsal", "unpatterned", "ridged", "whorl"]

DONOR_PALETTE = {
    "donor1": "#332288",
    "donor2": "#CC6677",
    "donor3": "#44AA99",
    "donor4": "#88CCEE",
    "donor5": "#DDCC77",
}

# TISSUE_PALETTE = {
#     "back":   "#117733",
#     "front":  "#AA3377",
#     "dorsal": "#44AA99",
#     "volar":  "#CC3311",
# }
TISSUE_PALETTE = {
    "back":   "#009E73",  # vivid green
    "front":  "#CC79A7",  # vivid pink-magenta
    "dorsal": "#0072B2",  # strong blue
    "volar":  "#E69F00",  # strong orange
}
TISSUE_ORDER = ["back", "front", "dorsal", "volar"]

TIMEPOINT_ORDER = ["12wk", "13wk", "14wk"]
TIMEPOINT_PALETTE = {
    "12wk": "#D3612C",  # orange
    "13wk": "#B23A5B",  # raspberry
    "14wk": "#2B3A67",  # navy (latest)
}


# Shape cluster palette — reused when cross-coloring niche UMAP by shape cluster
_CLUSTER_BASE = [
    "#0077BB", "#EE7733", "#009988", "#CC3311", "#EE3377",
    "#33BBEE", "#AA3377", "#BB5566", "#44AA99", "#DDCC77",
    "#882255", "#117733", "#88CCEE", "#332288", "#CC6677",
    "#EE9944", "#99DDBB", "#AA4499", "#661100", "#6699CC",
    "#888888", "#D4B483",
]


# Niche cluster palette — deliberately warmer/cooler than shape base so dual
# napari layers are visually distinguishable at a glance.
_NICHE_BASE = [
    "#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2",
    "#D55E00", "#CC79A7", "#999999", "#661100", "#332288",
    "#AA4466", "#6699CC", "#44AA99", "#DDCC77", "#117733",
    "#88CCEE", "#882255", "#CC6677", "#AA3377", "#FFB347",
    "#B87DF0", "#7DC37D",
]


def build_cluster_cmap(cluster_ids, base=None):
    """{int(cluster_id): hex_color}. Cycles gracefully past len(base)."""
    palette = base if base is not None else _CLUSTER_BASE
    ids = sorted({int(c) for c in cluster_ids if c is not None and c == c})
    return {c: palette[i % len(palette)] for i, c in enumerate(ids)}


def _apply_umap_axes(fig, x_title="UMAP 1", y_title="UMAP 2"):
    axkw = dict(
        showgrid=True, gridcolor="rgba(200,200,200,0.25)", gridwidth=0.5,
        zeroline=False,
        showticklabels=False, ticks="",
        showline=False,
        title_font=dict(size=FONT_AXIS_TITLE, family=FONT_FAMILY),
    )
    fig.update_xaxes(title_text=x_title, **axkw)
    fig.update_yaxes(title_text=y_title, **axkw)


def _base_layout(width, height, legend_title=None, show_legend=True):
    return dict(
        plot_bgcolor="#fff", paper_bgcolor="#fff",
        width=width, height=height,
        margin=dict(l=65, r=180, t=30, b=60),  # r=180 so external legends don't clip in SVG
        font=dict(family=FONT_FAMILY),
        showlegend=show_legend,
        legend=dict(
            font=dict(size=FONT_LEGEND, family=FONT_FAMILY),
            title=dict(
                text=legend_title or "",
                font=dict(size=FONT_LEGEND_TITLE, family=FONT_FAMILY),
            ),
            tracegroupgap=4,
            x=1.01, y=0.98, xanchor="left",
            bgcolor="rgba(255,255,255,0.9)",
            itemsizing="constant",
        ),
    )


def _ensure_sample_short(df):
    if "sample_short" not in df.columns:
        df = df.copy()
        df["sample_short"] = df["stack_name"].map(SHORT)
    return df


def _hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))


def _readable_text_color(hex_color, light="#FFFFFF", dark="#1A1A1A",
                        threshold=145):
    """Pick light or dark text for legibility on a colored fill.
    Uses ITU-R BT.601 luma. threshold=145 keeps white on most saturated
    colors but flips to dark on yellows / pale tones."""
    r, g, b = _hex_to_rgb(hex_color)
    luma = 0.299 * r + 0.587 * g + 0.114 * b
    return dark if luma > threshold else light


def apply_transform(values, method="raw", winsorize_sigma=5.0):
    """
    Feature transform matching the EpiCycle GUI (see
    napari_epicycle/segmentation/feature_stats.py:179-226).
    Preserves NaN.

    method : 'raw' | 'log1p' | 'sqrt' | 'rank' | 'zscore' | 'winsorize'

    Notes
    -----
    - log1p / sqrt shift by min if any finite value is negative so the
      transform is always defined.
    - rank normalizes to (0, 1] via scipy.stats.rankdata(x) / n.
    - winsorize clips to +/- winsorize_sigma std-devs then unscales.
    """
    arr = np.asarray(values, dtype=np.float64).copy()
    finite_mask = np.isfinite(arr)
    if not finite_mask.any() or method == "raw":
        return arr
    finite = arr[finite_mask]

    if method in ("log", "log1p"):
        if finite.min() < 0:
            arr[finite_mask] = arr[finite_mask] - finite.min()
            finite = arr[finite_mask]
        arr[finite_mask] = np.log1p(finite)
        return arr

    if method == "sqrt":
        if finite.min() < 0:
            arr[finite_mask] = arr[finite_mask] - finite.min()
            finite = arr[finite_mask]
        arr[finite_mask] = np.sqrt(finite)
        return arr

    if method == "rank":
        from scipy.stats import rankdata
        arr[finite_mask] = rankdata(finite) / len(finite)
        return arr

    if method == "zscore":
        if len(finite) > 1:
            mu = np.mean(finite)
            std = np.std(finite, ddof=1)
            if std > 0:
                arr[finite_mask] = (finite - mu) / std
        return arr

    if method == "winsorize":
        if len(finite) > 1:
            mu = np.mean(finite)
            std = np.std(finite, ddof=1)
            if std > 0:
                z = (finite - mu) / std
                z = np.clip(z, -winsorize_sigma, winsorize_sigma)
                arr[finite_mask] = z * std + mu
        return arr

    raise ValueError(f"Unknown transform method: {method!r}")


# -----------------------------------------------------------------------
# Sample metadata -- the SAME map as notebook 01.
# 'back' and 'front' skin are regions of their own (not 'dorsal' / 'volar').
# Expected stacks per tissue: volar 5, dorsal 3, back 3, front 2.
# apply_metadata() re-stamps donor / timepoint / tissue / pattern from this
# map, so a CSV or pickle written before the change still gets the right
# labels (and the right overlay colours).
# -----------------------------------------------------------------------
META_MAP = {
    '(14 wks) digit 5 unpatterned skin x40 stack-Smooth05_Resampled.tif':
        {'donor': 'donor1', 'timepoint': '14wk', 'tissue': 'volar', 'pattern': 'unpatterned'},
    '(13 wks) D3L ventral ridges x40-Smooth05_Resampled.tif':
        {'donor': 'donor2', 'timepoint': '13wk', 'tissue': 'volar', 'pattern': 'ridged'},
    '(13 wks) D3L Ventral unpatterned skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor2', 'timepoint': '13wk', 'tissue': 'volar', 'pattern': 'unpatterned'},
    '(13 wks) D3L Dorsal skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor2', 'timepoint': '13wk', 'tissue': 'dorsal', 'pattern': 'dorsal'},
    '(13 wks) Back skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor2', 'timepoint': '13wk', 'tissue': 'back', 'pattern': 'back'},
    '(13 wks) Front skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor2', 'timepoint': '13wk', 'tissue': 'front', 'pattern': 'front'},
    '(14 wks) ventral whorl x40-Smooth05_Resampled.tif':
        {'donor': 'donor3', 'timepoint': '14wk', 'tissue': 'volar', 'pattern': 'whorl'},
    '(14 wks) back skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor3', 'timepoint': '14wk', 'tissue': 'back', 'pattern': 'back'},
    '(14 wks) D3L dorsal x40-Smooth05_Resampled.tif':
        {'donor': 'donor3', 'timepoint': '14wk', 'tissue': 'dorsal', 'pattern': 'dorsal'},
    '(12 wks) D2L ventral tip x40-Smooth05_Resampled.tif':
        {'donor': 'donor4', 'timepoint': '12wk', 'tissue': 'volar', 'pattern': 'ridged'},
    '(12 wks) back skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor4', 'timepoint': '12wk', 'tissue': 'back', 'pattern': 'back'},
    '(12 wks) front skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor4', 'timepoint': '12wk', 'tissue': 'front', 'pattern': 'front'},
    '(12 wks) D3L dorsal x40 stack-Smooth05_Resampled.tif':
        {'donor': 'donor5', 'timepoint': '12wk', 'tissue': 'dorsal', 'pattern': 'dorsal'},
}
META_KEYS = ("donor", "timepoint", "tissue", "pattern")


def apply_metadata(df):
    """Re-stamp donor / timepoint / tissue / pattern from META_MAP.
    Prints how many rows changed per column and any stack not in the map
    (those rows keep whatever labels they had)."""
    df = df.copy()
    unmapped = sorted(set(df["stack_name"].unique()) - set(META_MAP))
    for key in META_KEYS:
        new = df["stack_name"].map(lambda s, k=key: META_MAP.get(s, {}).get(k))
        old = df[key] if key in df.columns else None
        keep = new.isna()
        if old is not None:
            new = new.where(~keep, old)
            n_changed = int((new.astype(str) != old.astype(str)).sum())
        else:
            n_changed = len(df)
        df[key] = new
        if n_changed:
            print(f"[metadata] {key}: {n_changed:,} rows relabelled from META_MAP")
    if unmapped:
        print(f"[metadata] WARNING stacks not in META_MAP (labels left as they were): {unmapped}")
    for key in ("tissue", "pattern"):
        stacks = df.groupby(key)["stack_name"].nunique()
        print(f"[metadata] {key}: " + ", ".join(
            f"{v} {stacks[v]} stacks / {n:,} cells"
            for v, n in df[key].value_counts().items()))
    return df


# -----------------------------------------------------------------------
# Grouped scatter -- how the points of different groups are layered
#   "intermix" : every point in ONE layer, in a random (seeded) order, so no
#                group sits on top of another.  Default.
#   "stacked"  : one layer per group, drawn group after group (the old look:
#                the group drawn last covers the others where they overlap).
# Every categorical UMAP function takes draw= / seed= to override per figure.
# -----------------------------------------------------------------------
SCATTER_DRAW = "intermix"
SCATTER_SEED = 0


def _ordered_present(values, order):
    """`order` filtered to the values present, then any value missing from
    `order` (sorted) -- so a group is never silently dropped from a plot."""
    present = set(values)
    extras = sorted(present - set(order), key=str)
    if extras:
        print(f"[plot] not in the order list, drawn last: {extras}")
    return [v for v in order if v in present] + extras


def _grouped_scatter(fig, d, x_col, y_col, group_col, groups, colors, *,
                     names=None, hover=None, legendgroups=None,
                     marker_size=6, opacity=0.8,
                     draw=None, seed=None, biggest_first=True):
    """Add a categorical scatter of `d` to `fig`.

    groups       : group values, in legend order (other values are skipped)
    colors       : {group: hex}; a group missing from it is grey (#888)
    names        : {group: legend text}            (default str(group))
    hover        : {group: hover text, no <extra>} (default "<b>name</b><br>n = N")
    legendgroups : {group: plotly legendgroup}     (optional)
    draw         : "intermix" | "stacked"           (default SCATTER_DRAW)
    seed         : shuffle seed for "intermix"      (default SCATTER_SEED)
    biggest_first: "stacked" only -- draw the biggest group first, so small
                   groups end up on top (as before); False = legend order.

    In "intermix" the points are one trace and the legend entries are
    empty marker-only traces, so clicking a legend entry does not hide its
    points (use draw="stacked" for that).
    """
    draw = SCATTER_DRAW if draw is None else draw
    seed = SCATTER_SEED if seed is None else seed
    if draw not in ("intermix", "stacked"):
        raise ValueError(f"draw must be 'intermix' or 'stacked', got {draw!r}")
    names, hover, legendgroups = names or {}, hover or {}, legendgroups or {}

    counts = d[group_col].value_counts()
    groups = [g for g in groups if counts.get(g, 0) > 0]
    name_of = {g: names.get(g, str(g)) for g in groups}
    hover_of = {g: hover.get(g, f"<b>{name_of[g]}</b><br>n = {int(counts[g]):,}")
                for g in groups}
    base = dict(size=marker_size, opacity=opacity, line=dict(width=0))
    tmpl = "%{x:.2f}, %{y:.2f}<extra></extra>"

    if draw == "stacked":
        order = (sorted(groups, key=lambda g: -counts[g]) if biggest_first
                 else list(groups))
        for g in order:
            sub = d[d[group_col] == g]
            fig.add_trace(go.Scatter(
                x=sub[x_col], y=sub[y_col], mode="markers",
                marker=dict(base, color=colors.get(g, "#888")),
                name=name_of[g], legendgroup=legendgroups.get(g),
                legendrank=groups.index(g),
                hovertemplate=hover_of[g] + "<br>" + tmpl,
            ))
        return fig

    # intermix: one shuffled layer + legend-only entries
    sub = d[d[group_col].isin(groups)]
    sub = sub.iloc[np.random.default_rng(seed).permutation(len(sub))]
    grp = sub[group_col]
    fig.add_trace(go.Scatter(
        x=sub[x_col].to_numpy(), y=sub[y_col].to_numpy(), mode="markers",
        marker=dict(base, color=grp.map(lambda g: colors.get(g, "#888")).to_numpy()),
        customdata=grp.map(hover_of).to_numpy(),
        hovertemplate="%{customdata}<br>" + tmpl,
        name="all points", showlegend=False,
    ))
    for rank, g in enumerate(groups):
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers",
            marker=dict(base, color=colors.get(g, "#888")),
            name=name_of[g], legendgroup=legendgroups.get(g),
            legendrank=rank, hoverinfo="skip", showlegend=True,
        ))
    return fig


print("Setup loaded:", len(STACK_ORDER), "samples,",
      len(PATTERN_ORDER), "patterns,", len(DONOR_PALETTE), "donors")



# -----------------------------------------------------------------------
# Cluster-color registry -- single source of truth for cluster colors.
#
# Why this exists: the gallery, the napari label layers, the adjacency
# edges, and the exploded view each used to build their own cmaps from
# different cluster sets (whole-df vs per-stack vs per-config). Because
# build_cluster_cmap assigns colors via sorted-enumerate, any stack with
# a missing cluster ID would silently re-number the palette and the
# napari layer's "cluster 1" would not match the gallery's "cluster 1".
#
# These helpers always read from the dataset-wide cluster set (df for
# shape, niche_results_dict for niche) so cluster N maps to the same
# color regardless of the stack, config, or section that's rendering.
# Resolution swaps are picked up automatically on the next call.
# -----------------------------------------------------------------------

def get_shape_cmap(resolution=None):
    """Canonical {cluster_id: hex} for shape clusters at `resolution`
    (defaults to SHAPE_RESOLUTION).
    Reads from df, the dataset-wide source of truth for shape clustering.
    """
    res = resolution if resolution is not None else SHAPE_RESOLUTION
    col = f"leiden_r{res}"
    if "df" not in globals() or col not in df.columns:
        return build_cluster_cmap([], base=_CLUSTER_BASE)
    return build_cluster_cmap(df[col].dropna().astype(int), base=_CLUSTER_BASE)


def get_niche_cmap(config=None, resolution=None):
    """Canonical {cluster_id: hex} for a (config, resolution) niche pair
    (defaults to NICHE_CONFIG, NICHE_RESOLUTION).
    Reads from niche_results_dict directly so it works for non-featured
    configs too -- those don't have columns merged into df.
    """
    cfg = config if config is not None else NICHE_CONFIG
    res = resolution if resolution is not None else NICHE_RESOLUTION
    if "niche_results_dict" not in globals():
        return build_cluster_cmap([], base=_NICHE_BASE)
    nrdata = niche_results_dict.get(cfg)
    if nrdata is None or res not in nrdata.get("clusters", {}):
        return build_cluster_cmap([], base=_NICHE_BASE)
    membership = nrdata["clusters"][res]["membership"]
    return build_cluster_cmap(np.asarray(membership).astype(int),
                              base=_NICHE_BASE)


## A4-aware figure export (run once)

`save_fig(fig, name, mm=...)` writes an Illustrator-editable **SVG** (vector) and
a high-DPI **PNG** sidecar. The `mm` argument sizes the figure for an **A4
Illustrator artboard** at 100%. Requires `kaleido` (`pip install -U kaleido`).

See the *Saving figures* section below for usage details.


In [ ]:
import os

# ======================================================================
# A4-aware Plotly figure export
#
# Targets an A4 (210 x 297 mm) Illustrator artboard as the max canvas.
# Illustrator imports SVG at 72 dpi, so sizing in mm at 72 dpi means a
# placement at 100% inside an A4 artboard renders at the true physical
# size you specified.
# ======================================================================
MM_PER_INCH = 25.4
SVG_DPI = 72            # Illustrator SVG import dpi
PNG_DPI = 300           # sidecar raster preview

def mm_to_px(mm, dpi=SVG_DPI):
    return int(round(mm * dpi / MM_PER_INCH))

# Physical-size presets (max footprint = A4 portrait, 210 x 297 mm)
A4_PORTRAIT_MM  = (210, 297)
A4_LANDSCAPE_MM = (297, 210)
PANEL_FULL_MM   = (180, 130)   # full-width manuscript panel
PANEL_HALF_MM   = (86, 70)     # half-width / single-column panel

FIG_OUT_DIR = 'figures'


def save_fig(fig, name, *, mm=None, width_px=None, height_px=None,
             out_dir=None, formats=("svg", "png")):
    """Save a Plotly figure as SVG (vector, Illustrator-editable) + high-dpi PNG.

    Sizing:
        mm=(w, h)            -> sized to those mm at 72 dpi (preferred).
        width_px / height_px -> raw pixel size.
        neither              -> uses fig.layout.width / height.

    PNG is rendered at PNG_DPI / SVG_DPI (~4.17x) for crisp previews.
    Requires kaleido:  pip install -U kaleido
    If the static image backend (kaleido) is missing, writes an interactive HTML instead.
    """
    out_dir = out_dir if out_dir is not None else FIG_OUT_DIR
    os.makedirs(out_dir, exist_ok=True)
    if mm is not None:
        width_px = mm_to_px(mm[0])
        height_px = mm_to_px(mm[1])
    png_scale = PNG_DPI / SVG_DPI

    paths = []
    try:
        for fmt in formats:
            path = os.path.join(out_dir, f"{name}.{fmt}")
            kw = {}
            if width_px:  kw["width"] = width_px
            if height_px: kw["height"] = height_px
            if fmt == "png":
                kw["scale"] = png_scale
            fig.write_image(path, format=fmt, **kw)
            paths.append(path)
            print(f"  -> {path}")
    except Exception as e:
        path = os.path.join(out_dir, f"{name}.html")
        fig.write_html(path, include_plotlyjs="cdn")
        print(f"  [no static export: {type(e).__name__}] -> {path}")
        return [path]
    return paths


## 2. Data loading

Loads `morphospace_results.pkl` + `niche_results.pkl`, selects **one niche
config** to feature in the static figure panels, and merges that config's
niche coordinates + cluster memberships into `df` as new columns
(`niche_umap_1`, `niche_umap_2`, `niche_leiden_r{res}`).

Change `NICHE_CONFIG` at the top of the cell to feature a different
configuration. All downstream panels refresh on rerun. The full niche dict
is preserved as `niche_results_dict` for cross-config comparisons (used by
the napari launcher at the bottom).


In [ ]:
# ========== USER KNOBS ==========
MORPHO_OUTPUT_DIR = r'run'                              # where pkls + rep CSVs live
NICHE_CONFIG      = "hop3_pure"                         # which niche config to feature in static panels
SHAPE_RESOLUTION  = 1.0                                  # leiden_r{SHAPE_RESOLUTION} in df
NICHE_RESOLUTION  = 1.5                                 # niche clustering resolution to use


# --- Mesh gallery sizing ---
# True  -> shared scale across the gallery: the largest representative cell
#          fills its panel, smaller cells render proportionally smaller
#          (preserves real cell-size differences between clusters).
# False -> per-cell normalization: every cell is rescaled to fill its own
#          panel (better for comparing pure shape, but loses size cue).
RELATIVE_SIZES  = False # True
# ================================




# -- Load shape morphospace --
morpho_pkl = os.path.join(MORPHO_OUTPUT_DIR, "morphospace_results.pkl")
with open(morpho_pkl, "rb") as f:
    morpho = pickle.load(f)

df       = morpho["df"].copy()
df       = apply_metadata(df)   # back skin = its own region (META_MAP)
pcs      = morpho["pcs"]
# Expose each shape PC as a df column (PC1, PC2, ...) so it can be
# passed to umap_by_feature(df, "PC1", ...) like any other numeric column.
if pcs is not None:
    for _k in range(pcs.shape[1]):
        df[f"PC{_k + 1}"] = pcs[:, _k]
mesh_lookup_shape = morpho.get("mesh_lookup", {})

# -- Load niche results --
niche_pkl = os.path.join(MORPHO_OUTPUT_DIR, "niche_results.pkl")
with open(niche_pkl, "rb") as f:
    niche_data = pickle.load(f)

niche_results_dict = niche_data["niche_results"]
NICHE_LEIDEN_RES   = niche_data.get("NICHE_LEIDEN_RES", [0.5, 1.0, 1.5])
NICHE_CONFIGS      = niche_data.get("NICHE_CONFIGS", [])

print("Available niche configs:", list(niche_results_dict.keys()))
print("Niche leiden resolutions:", NICHE_LEIDEN_RES)
print()


# -- Merge chosen niche config into df --
if NICHE_CONFIG not in niche_results_dict:
    raise KeyError(
        f"Niche config {NICHE_CONFIG!r} not in results. "
        f"Available: {list(niche_results_dict.keys())}"
    )

nr          = niche_results_dict[NICHE_CONFIG]
valid_mask  = nr["valid_mask"]
niche_umap  = nr["umap"]             # (N_valid, 2)

# Full-length arrays, NaN where not valid
full_umap = np.full((len(df), 2), np.nan)
full_umap[valid_mask] = niche_umap
df["niche_umap_1"] = full_umap[:, 0]
df["niche_umap_2"] = full_umap[:, 1]

# Expose niche PCs from the SAME featured config as df columns.
# These were the input to the niche UMAP, so naming and indexing match.
_niche_pcs_arr = nr.get("niche_pcs")
if _niche_pcs_arr is not None:
    _full_npcs = np.full((len(df), _niche_pcs_arr.shape[1]), np.nan)
    _full_npcs[valid_mask] = _niche_pcs_arr[valid_mask]
    for _k in range(_niche_pcs_arr.shape[1]):
        df[f"niche_PC{_k + 1}"] = _full_npcs[:, _k]
    print(
        f"Niche PCs from {NICHE_CONFIG}: niche_PC1..niche_PC{_niche_pcs_arr.shape[1]} "
        f"({valid_mask.sum():,}/{len(df):,} cells)"
    )
else:
    print(f"[WARN] no niche_pcs key in niche_results[{NICHE_CONFIG!r}] — niche_PCn columns skipped")

# Niche cluster memberships at every resolution — float w/ NaN for invalid
for res in NICHE_LEIDEN_RES:
    col = f"niche_leiden_r{res}"
    full_m = np.full(len(df), np.nan, dtype=float)
    full_m[valid_mask] = nr["clusters"][res]["membership"].astype(float)
    df[col] = full_m

n_valid = int(valid_mask.sum())
print(f"Niche config: {NICHE_CONFIG}  ({n_valid:,}/{len(df):,} cells have adjacency)")
print(f"Feature cols added: niche_umap_1, niche_umap_2, "
      + ", ".join(f"niche_leiden_r{r}" for r in NICHE_LEIDEN_RES))

# Sanity check on shape cluster col
shape_leiden_col = f"leiden_r{SHAPE_RESOLUTION}"
if shape_leiden_col not in df.columns:
    raise KeyError(f"{shape_leiden_col!r} missing from morphospace df")

# Build canonical cluster palettes
SHAPE_CLUSTER_CMAP = get_shape_cmap(SHAPE_RESOLUTION)
NICHE_CLUSTER_CMAP = get_niche_cmap(NICHE_CONFIG, NICHE_RESOLUTION)
print(f"Shape clusters:  {len(SHAPE_CLUSTER_CMAP)}")
print(f"Niche clusters:  {len(NICHE_CLUSTER_CMAP)}")


## Step 0 -- label sanity check

In [ ]:
# Step 0 -- label sanity check.
# The sync itself is already in this notebook: the setup cell defines META_MAP,
# apply_metadata(), TISSUE_PALETTE / TISSUE_ORDER (back, front, dorsal, volar),
# PATTERN_PALETTE / PATTERN_ORDER (with back, front), SAMPLE_PALETTE and
# STACK_ORDER exactly as notebook 03, and the loading cell above runs
# df = apply_metadata(df).  This cell only verifies it, and raises if it is off.
_expected = {"volar": 5, "dorsal": 3, "back": 3, "front": 2}
_got = df.groupby("tissue")["stack_name"].nunique().to_dict()
print("[Step 0] stacks per tissue:", _got)
if _got != _expected:
    raise ValueError(f"[Step 0] stacks per tissue {_got} != {_expected}")
for _k in ("back", "front", "dorsal", "volar"):
    assert _k in TISSUE_PALETTE and _k in TISSUE_ORDER, f"TISSUE_PALETTE/ORDER lacks {_k!r}"
for _k in ("back", "front"):
    assert _k in PATTERN_PALETTE and _k in PATTERN_ORDER, f"PATTERN_PALETTE/ORDER lacks {_k!r}"
assert "save_fig" in globals(), "save_fig missing -- run the A4 export cell"
_no_niche = df.loc[df["niche_umap_1"].isna()].groupby("stack_name").size()
print("[Step 0] cells without niche coordinates per stack:",
      _no_niche.to_dict() if len(_no_niche) else "none")

## 3. Plot functions (reusable)

All UMAP functions accept `x_col` / `y_col` kwargs — pass
`"niche_umap_1"` / `"niche_umap_2"` for niche-embedding views, or leave
defaults for shape views. `umap_by_cluster` additionally accepts a
`cluster_col` override (for cross-embedding colorings).

`umap_by_feature(df, feature, ...)` is new — continuous viridis with
percentile clipping.

`violin_panel(df, feature, group_col, ...)` produces a single half-violin
+ narrow box + side-jittered scatter panel across the categories of
`group_col` (aesthetic matches `figure2_main.py`). Fully parameterized.

Run this cell once.


In [ ]:
# =======================================================================
# UMAP — categorical colorings (each accepts x_col / y_col overrides)
# =======================================================================
# =======================================================================
# Shared helper: format cluster labels (int-valued floats -> int strings)
# =======================================================================
def _fmt_cluster(v):
    """Format a cluster label: integer-valued floats become int strings.

    Niche cluster columns are stored as float so they can hold NaN for
    cells outside the valid mask. Without this helper, str(1.0) shows as
    "1.0" which looks wrong in tick labels.
    """
    if v is None:
        return "NA"
    try:
        import math as _math
        f = float(v)
        if _math.isnan(f):
            return "NA"
        if f.is_integer():
            return str(int(f))
    except (TypeError, ValueError):
        pass
    return str(v)


def _umap_titles(x_col, y_col):
    return dict(x_title="Niche UMAP 1" if x_col.startswith("niche_") else "UMAP 1",
                y_title="Niche UMAP 2" if y_col.startswith("niche_") else "UMAP 2")


# All categorical UMAPs draw through _grouped_scatter (helpers cell):
# draw="intermix" (default, SCATTER_DRAW) or draw="stacked" (the old look).
def umap_by_sample(df, marker_size=6, opacity=0.8, show_legend=True,
                   width=1000, height=700,
                   x_col="umap_1", y_col="umap_2", draw=None, seed=None):
    df = _ensure_sample_short(df)
    d = df.dropna(subset=[x_col, y_col, "sample_short"])
    order = _ordered_present(d["sample_short"].unique(),
                             [SHORT[s] for s in STACK_ORDER])
    fig = go.Figure()
    _grouped_scatter(fig, d, x_col, y_col, "sample_short", order, SAMPLE_PALETTE,
                     marker_size=marker_size, opacity=opacity, draw=draw, seed=seed)
    fig.update_layout(**_base_layout(width, height, legend_title="<b>Sample</b>"))
    _apply_umap_axes(fig, **_umap_titles(x_col, y_col))
    fig.update_layout(showlegend=show_legend)
    return fig


def umap_by_pattern(df, marker_size=6, opacity=0.75, show_legend=True,
                    width=900, height=680,
                    x_col="umap_1", y_col="umap_2", draw=None, seed=None):
    d = df.dropna(subset=[x_col, y_col, "pattern"])
    fig = go.Figure()
    _grouped_scatter(fig, d, x_col, y_col, "pattern",
                     _ordered_present(d["pattern"].unique(), PATTERN_ORDER),
                     PATTERN_PALETTE, marker_size=marker_size, opacity=opacity,
                     draw=draw, seed=seed)
    fig.update_layout(**_base_layout(width, height, legend_title="<b>Pattern</b>"))
    _apply_umap_axes(fig, **_umap_titles(x_col, y_col))
    fig.update_layout(showlegend=show_legend)
    return fig


def umap_by_donor(df, marker_size=6, opacity=0.75, show_legend=True,
                  width=900, height=680,
                  x_col="umap_1", y_col="umap_2", draw=None, seed=None):
    d = df.dropna(subset=[x_col, y_col, "donor"])
    fig = go.Figure()
    _grouped_scatter(fig, d, x_col, y_col, "donor",
                     sorted(d["donor"].unique(), key=str), DONOR_PALETTE,
                     marker_size=marker_size, opacity=opacity, draw=draw, seed=seed)
    fig.update_layout(**_base_layout(width, height, legend_title="<b>Donor</b>"))
    _apply_umap_axes(fig, **_umap_titles(x_col, y_col))
    fig.update_layout(showlegend=show_legend)
    return fig


def umap_by_tissue(df, marker_size=6, opacity=0.75, show_legend=True,
                   width=860, height=680,
                   x_col="umap_1", y_col="umap_2", draw=None, seed=None):
    d = df.dropna(subset=[x_col, y_col, "tissue"])
    fig = go.Figure()
    _grouped_scatter(fig, d, x_col, y_col, "tissue",
                     _ordered_present(d["tissue"].unique(), TISSUE_ORDER),
                     TISSUE_PALETTE, marker_size=marker_size, opacity=opacity,
                     draw=draw, seed=seed)
    fig.update_layout(**_base_layout(width, height, legend_title="<b>Tissue</b>"))
    _apply_umap_axes(fig, **_umap_titles(x_col, y_col))
    fig.update_layout(showlegend=show_legend)
    return fig


def umap_by_timepoint(df, marker_size=6, opacity=0.75, show_legend=True,
                      width=900, height=680,
                      x_col="umap_1", y_col="umap_2", draw=None, seed=None):
    d = df.dropna(subset=[x_col, y_col, "timepoint"])
    fig = go.Figure()
    _grouped_scatter(fig, d, x_col, y_col, "timepoint",
                     _ordered_present(d["timepoint"].unique(), TIMEPOINT_ORDER),
                     TIMEPOINT_PALETTE, marker_size=marker_size, opacity=opacity,
                     draw=draw, seed=seed)
    fig.update_layout(**_base_layout(width, height, legend_title="<b>Timepoint</b>"))
    _apply_umap_axes(fig, **_umap_titles(x_col, y_col))
    fig.update_layout(showlegend=show_legend)
    return fig


def umap_by_cluster(df, resolution=None, marker_size=6, opacity=0.85, show_legend=True,
                    width=900, height=700, label_centroids=False,
                    centroid_method="median",
                    x_col="umap_1", y_col="umap_2",
                    cluster_col=None, palette=None,
                    legend_title=None, draw=None, seed=None):
    """
    Colors by any cluster column. Pass cluster_col explicitly to override
    the default (`leiden_r{resolution}` for shape / `niche_leiden_r{resolution}`
    for niche depending on x_col).

    palette=None -> auto-picks SHAPE_CLUSTER_CMAP for shape clusters,
                    NICHE_CLUSTER_CMAP for niche clusters.
    """
    if cluster_col is None:
        if x_col.startswith("niche_"):
            cluster_col = f"niche_leiden_r{resolution if resolution else NICHE_RESOLUTION}"
        else:
            cluster_col = f"leiden_r{resolution if resolution else SHAPE_RESOLUTION}"
    if cluster_col not in df.columns:
        raise KeyError(f"Column {cluster_col!r} not in dataframe")

    d = df.dropna(subset=[x_col, y_col, cluster_col]).copy()
    d["_cluster"] = d[cluster_col].astype(int)
    clusters = sorted(d["_cluster"].unique())
    counts = d["_cluster"].value_counts()

    if palette is None:
        # Match palette to cluster column semantics
        if cluster_col.startswith("niche_"):
            palette = NICHE_CLUSTER_CMAP if cluster_col == f"niche_leiden_r{NICHE_RESOLUTION}" \
                else build_cluster_cmap(clusters, base=_NICHE_BASE)
        else:
            palette = SHAPE_CLUSTER_CMAP if cluster_col == f"leiden_r{SHAPE_RESOLUTION}" \
                else build_cluster_cmap(clusters, base=_CLUSTER_BASE)

    fig = go.Figure()
    _grouped_scatter(fig, d, x_col, y_col, "_cluster", clusters, palette,
                     names={c: str(int(c)) for c in clusters},
                     hover={c: f"<b>Cluster {int(c)}</b><br>n = {int(counts[c]):,}"
                            for c in clusters},
                     marker_size=marker_size, opacity=opacity,
                     draw=draw, seed=seed, biggest_first=False)

    if label_centroids:
        for c in clusters:
            sub = d[d["_cluster"] == c]
            if centroid_method == "median":
                cx, cy = sub[x_col].median(), sub[y_col].median()
            else:
                cx, cy = sub[x_col].mean(), sub[y_col].mean()
            fig.add_annotation(
                x=cx, y=cy, text=f"<b>{int(c)}</b>", showarrow=False,
                font=dict(size=12, color="#222", family=FONT_FAMILY),
                bgcolor="rgba(255,255,255,0.78)",
                bordercolor="rgba(0,0,0,0.22)",
                borderwidth=0.6, borderpad=2.5,
            )

    lt = legend_title or (
        "<b>Niche cluster</b>" if cluster_col.startswith("niche_") else "<b>Cluster</b>"
    )
    fig.update_layout(**_base_layout(width, height, legend_title=lt))
    _apply_umap_axes(fig, **_umap_titles(x_col, y_col))
    fig.update_layout(showlegend=show_legend)
    return fig


# =======================================================================
# UMAP — continuous feature coloring
# =======================================================================
def umap_by_feature(df, feature, show_legend=True,
                    x_col="umap_1", y_col="umap_2",
                    colorscale="Viridis",
                    pct_clip=(1, 99),
                    marker_size=5, opacity=0.75,
                    width=920, height=700,
                    transform="raw",
                    log_transform=None,
                    colorbar_title=None,
                    reverse_scale=False):
    """
    UMAP colored by any numeric feature column. Robust to outliers via
    percentile clipping (default 1-99).

    transform : 'raw' | 'log1p' | 'sqrt' | 'rank' | 'zscore' | 'winsorize'
                Matches the EpiCycle GUI transform set. 'rank' is especially
                useful for non-comparable features (maps to (0, 1]).

    log_transform : bool, deprecated
                    Back-compat alias -- True maps to transform='log1p'.
    """
    if feature not in df.columns:
        raise KeyError(f"Feature column {feature!r} not in df")
    if log_transform is True:
        transform = "log1p"

    d = df.dropna(subset=[x_col, y_col, feature])
    vals = apply_transform(d[feature].astype(float).values, method=transform)
    lo, hi = np.percentile(vals, pct_clip)
    # Guard against constant features
    if hi <= lo:
        hi = lo + 1e-9

    suffix = "" if transform == "raw" else f" ({transform})"
    label = colorbar_title or (feature + suffix)

    fig = go.Figure(go.Scatter(
        x=d[x_col], y=d[y_col], mode="markers",
        marker=dict(
            size=marker_size, color=vals,
            colorscale=colorscale, cmin=lo, cmax=hi,
            reversescale=reverse_scale,
            opacity=opacity, line=dict(width=0),
            colorbar=dict(
                title=dict(text=label,
                           font=dict(size=FONT_AXIS_TITLE - 1, family=FONT_FAMILY)),
                thickness=16, len=0.72,
                tickfont=dict(size=FONT_TICK - 1, family=FONT_FAMILY),
                outlinewidth=0,
            ),
        ),
        hovertemplate=f"<b>{feature}</b>: %{{marker.color:.3g}}<extra></extra>",
        showlegend=False,
    ))
    fig.update_layout(**_base_layout(width, height, show_legend=False))
    _apply_umap_axes(fig,
                     x_title="Niche UMAP 1" if x_col.startswith("niche_") else "UMAP 1",
                     y_title="Niche UMAP 2" if y_col.startswith("niche_") else "UMAP 2")
    fig.update_layout(showlegend=show_legend)
    return fig


# =======================================================================
# Violin — single feature across one categorical grouping
#   aesthetic matches figure2_main.py: half-violin + narrow box + side jitter
# =======================================================================
def violin_panel(df, feature, group_col, show_legend=True,
                 palette=None, group_order=None,
                 show_box=True, show_points=True, max_points=200,
                 show_median_line=False, baseline=None,
                 violin_width=0.6, box_width=0.3, jitter_frac=0.12,
                 violin_fill_alpha=0.08, violin_line_alpha=0.4,
                 point_size=3.5, point_opacity=0.32,
                 y_transform=None,
                 width=760, height=500,
                 y_title=None, x_title=None,
                 tick_rotation=0,
                 show_n_annotation=True,
                 include_zero=True,
                 random_seed=42):
    """
    Single-feature violin across the categories in `group_col`.
    Aesthetic: half-violin + overlaid narrow box + jittered scatter (side-offset),
    optional median trend line + baseline reference.

    Parameters
    ----------
    df : DataFrame
    feature : str  — numeric column to plot on y
    group_col : str  — categorical column; one violin per value
    palette : dict {group_value: hex_color} or None (auto)
    group_order : list of group values in display order (default: sorted)
    show_box / show_points : toggle overlay elements
    max_points : subsample threshold for the jitter scatter
    show_median_line : connect group medians (dashed red) — useful for
                      ordered groups like timepoint, off for Leiden clusters
    baseline : optional horizontal reference line (y-value)
    violin_width / box_width / jitter_frac : geometry controls
    violin_fill_alpha / violin_line_alpha / point_opacity : visual weights
    y_transform : None | 'raw' | 'log1p' | 'sqrt' | 'rank' | 'zscore' | 'winsorize'
        Matches the EpiCycle GUI transform set (feature_stats.apply_transform).
    tick_rotation : degrees on x labels (useful for sample names)
    """
    import numpy as _np
    d = df[[feature, group_col]].dropna().copy()

    if y_transform and y_transform != "raw":
        d[feature] = apply_transform(d[feature].values, method=y_transform)

    if group_order is None:
        try:
            group_order = sorted(d[group_col].unique(),
                                 key=lambda x: float(x))
        except (TypeError, ValueError):
            group_order = sorted(d[group_col].unique(), key=str)

    if palette is None:
        palette = {g: _CLUSTER_BASE[i % len(_CLUSTER_BASE)]
                   for i, g in enumerate(group_order)}

    rng = _np.random.default_rng(random_seed)
    fig = go.Figure()
    legend_added_median = False

    for i, g in enumerate(group_order):
        sub = d[d[group_col] == g][feature]
        if sub.empty:
            continue
        c_hex = palette.get(g, "#888888")
        r, gr, b = _hex_to_rgb(c_hex)
        fill_rgba = f"rgba({r},{gr},{b},{violin_fill_alpha:.3f})"
        line_rgba = f"rgba({r},{gr},{b},{violin_line_alpha:.3f})"

        # Half-violin (positive side)
        fig.add_trace(go.Violin(
            y=sub, x0=i,
            side="positive", width=violin_width,
            line=dict(color=line_rgba, width=1),
            fillcolor=fill_rgba,
            meanline_visible=False,
            showlegend=False, hoverinfo="skip", points=False,
            scalemode="width",
        ))

        if show_box:
            fig.add_trace(go.Box(
                y=sub, x0=i, orientation="v",
                boxpoints=False, width=box_width, whiskerwidth=0.4,
                line=dict(color="#666666", width=1.0),
                fillcolor="rgba(100,100,100,0.04)",
                showlegend=False, hoverinfo="skip",
                marker=dict(opacity=0),
            ))

        if show_points:
            pts = sub.sample(max_points, random_state=random_seed) \
                if len(sub) > max_points else sub
            jitter = rng.uniform(-jitter_frac, jitter_frac, size=len(pts))
            fig.add_trace(go.Scatter(
                x=i + jitter - 0.15,
                y=pts.values, mode="markers",
                marker=dict(size=point_size, color=c_hex,
                            opacity=point_opacity, line=dict(width=0)),
                showlegend=False,
                hovertemplate=(
                    f"<b>{g}</b><br>"
                    f"{feature}: %{{y:.3f}}<extra></extra>"
                ),
            ))

    if show_median_line:
        medians = [d[d[group_col] == g][feature].median() for g in group_order]
        fig.add_trace(go.Scatter(
            x=list(range(len(group_order))), y=medians,
            mode="lines+markers",
            line=dict(color="#CC3311", width=1.5, dash="dash"),
            marker=dict(symbol="circle-open", size=7, color="#CC3311",
                        line=dict(width=1.5, color="#CC3311")),
            showlegend=True, name="Median",
            hovertemplate="Group: %{x}<br>Median: %{y:.3f}<extra></extra>",
        ))
        legend_added_median = True

    if baseline is not None:
        fig.add_shape(
            type="line",
            x0=-0.5, x1=len(group_order) - 0.5,
            y0=baseline, y1=baseline,
            line=dict(color="#999", width=1, dash="dashdot"),
        )

    fig.update_layout(
        plot_bgcolor="#fff", paper_bgcolor="#fff",
        width=width, height=height,
        margin=dict(l=70, r=20, t=30, b=80 if tick_rotation else 60),
        font=dict(family=FONT_FAMILY),
        showlegend=legend_added_median,
        legend=dict(
            font=dict(size=FONT_LEGEND, family=FONT_FAMILY),
            x=1.01, y=0.98, xanchor="left",
            bgcolor="rgba(255,255,255,0.9)",
        ),
    )
    fig.update_xaxes(
        tickvals=list(range(len(group_order))),
        ticktext=[_fmt_cluster(g) for g in group_order],
        title_text=x_title or group_col,
        title_font=dict(size=FONT_AXIS_TITLE, family=FONT_FAMILY),
        tickfont=dict(size=FONT_TICK, family=FONT_FAMILY),
        tickangle=tick_rotation,
        range=[-0.6, len(group_order) - 0.4],
        showgrid=True, gridcolor="rgba(200,200,200,0.3)", gridwidth=0.5,
        zeroline=False,
    )
    _y_suffix = "" if not y_transform or y_transform == "raw" else f" ({y_transform})"
    fig.update_yaxes(
        title_text=y_title or (feature + _y_suffix),
        title_font=dict(size=FONT_AXIS_TITLE, family=FONT_FAMILY),
        tickfont=dict(size=FONT_TICK, family=FONT_FAMILY),
        gridcolor="rgba(200,200,200,0.3)", gridwidth=0.5,
        zeroline=False,
        rangemode="tozero" if include_zero else "normal",
    )

    if show_n_annotation:
        fig.add_annotation(
            text=f"n = {len(d):,}",
            x=0.98, y=0.03, xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=11, color="#555", family=FONT_FAMILY),
            xanchor="right", yanchor="bottom",
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="rgba(200,200,200,0.5)",
            borderwidth=1, borderpad=4,
        )
    fig.update_layout(showlegend=show_legend)
    return fig


# =======================================================================
# Enrichment heatmaps (log2 observed/expected)
# =======================================================================
def enrichment_heatmap(df, row_col, col_col, show_legend=True, colorscale="RdBu_r",
                       row_order=None, col_order=None,
                       row_palette=None, col_palette=None,
                       clip=3.0, width=900, height=520,
                       title=None,
                       row_label_prefix="", col_label_prefix="",
                       x_title=None, y_title=None):
    """
    log2(observed / expected) enrichment heatmap. Pass pattern col + cluster
    col, or shape_cluster + niche_cluster for cross-embedding comparisons.
    """
    d = df[[row_col, col_col]].dropna()
    if row_order is None:
        try:
            row_order = sorted(d[row_col].unique(), key=lambda x: float(x))
        except (TypeError, ValueError):
            row_order = sorted(d[row_col].unique(), key=str)
    if col_order is None:
        try:
            col_order = sorted(d[col_col].unique(), key=lambda x: float(x))
        except (TypeError, ValueError):
            col_order = sorted(d[col_col].unique(), key=str)

    n_total = len(d)
    Z = np.zeros((len(row_order), len(col_order)))
    for i, r in enumerate(row_order):
        r_mask = (d[row_col] == r)
        n_r = r_mask.sum()
        for j, c in enumerate(col_order):
            c_mask = (d[col_col] == c)
            n_c = c_mask.sum()
            obs = (r_mask & c_mask).sum() / max(n_r, 1)
            exp = n_c / max(n_total, 1)
            Z[i, j] = np.log2(obs / (exp + 1e-9) + 1e-9)
    Z = np.clip(Z, -clip, clip)

    def _fmt(v, prefix):
        if isinstance(v, (int, float)) and not isinstance(v, bool):
            if float(v).is_integer():
                return f"{prefix}{int(v)}"
        return f"{prefix}{v}"

    row_labels = [_fmt(r, row_label_prefix) for r in row_order]
    col_labels = [_fmt(c, col_label_prefix) for c in col_order]

    fig = go.Figure(go.Heatmap(
        z=Z,
        x=col_labels, y=row_labels,
        colorscale=colorscale, zmid=0, zmin=-clip, zmax=clip,
        colorbar=dict(
            title=dict(text="log2(O/E)",
                       font=dict(size=FONT_AXIS_TITLE - 1, family=FONT_FAMILY)),
            thickness=16, len=0.78,
            tickfont=dict(size=FONT_TICK - 1, family=FONT_FAMILY),
            outlinewidth=0,
        ),
        hovertemplate=(
            f"{row_col}: %{{y}}<br>{col_col}: %{{x}}<br>"
            "log2(O/E): %{z:.2f}<extra></extra>"
        ),
    ))
    fig.update_xaxes(
        title_text=x_title or col_col, type="category",
        title_font=dict(size=FONT_AXIS_TITLE, family=FONT_FAMILY),
        tickfont=dict(size=FONT_TICK, family=FONT_FAMILY),
    )
    fig.update_yaxes(
        title_text=y_title or row_col, type="category",
        autorange="reversed",
        title_font=dict(size=FONT_AXIS_TITLE, family=FONT_FAMILY),
        tickfont=dict(size=FONT_TICK, family=FONT_FAMILY),
    )
    fig.update_layout(
        plot_bgcolor="#fff", paper_bgcolor="#fff",
        width=width, height=height,
        margin=dict(l=110, r=60, t=40 if title else 20, b=70),
        font=dict(family=FONT_FAMILY),
        title=dict(text=title, font=dict(size=13, family=FONT_FAMILY)) if title else None,
    )
    fig.update_layout(showlegend=show_legend)
    return fig


# =======================================================================
# Compositional stacked bar — cluster x covariate (batch-effect sanity check)
# =======================================================================
def composition_bar(df, cluster_col, covariate_col, show_legend=True,
                    palette=None, cov_order=None, cluster_order=None,
                    normalize="cluster",
                    width=1100, height=520, bargap=0.18,
                    legend_title=None,
                    x_title=None, y_title=None,
                    show_counts=True,
                    tick_rotation=0):
    """
    Stacked bar showing the covariate composition of each cluster.
    Primary use: batch-effect check -- if one donor/sample dominates a cluster,
    that cluster is probably batch-driven.

    Parameters
    ----------
    df : DataFrame
    cluster_col : cluster assignment column (e.g. "leiden_r1.0" or "niche_leiden_r1.0")
    covariate_col : categorical column to decompose (e.g. "donor", "sample_short",
                   "pattern", "tissue", "timepoint")
    palette : {cov_value: hex_color}. None -> auto-cycles through _CLUSTER_BASE.
    normalize : 'cluster'   -> each cluster bar sums to 1
                              ("what fraction of cluster X is each covariate?")
                'covariate' -> each covariate's contribution sums to 1
                              ("how does each covariate spread across clusters?")
    show_counts : annotate cluster size (n=N) above each bar when normalize='cluster'
    tick_rotation : degrees on x labels
    """
    d = df[[cluster_col, covariate_col]].dropna()
    if d.empty:
        raise ValueError(f"No rows with both {cluster_col!r} and {covariate_col!r}")

    if cluster_order is None:
        try:
            cluster_order = sorted(d[cluster_col].unique(), key=lambda x: float(x))
        except (TypeError, ValueError):
            cluster_order = sorted(d[cluster_col].unique(), key=str)

    if cov_order is None:
        try:
            cov_order = sorted(d[covariate_col].unique(), key=lambda x: float(x))
        except (TypeError, ValueError):
            cov_order = sorted(d[covariate_col].unique(), key=str)

    if palette is None:
        palette = {v: _CLUSTER_BASE[i % len(_CLUSTER_BASE)]
                   for i, v in enumerate(cov_order)}

    counts = d.groupby([cluster_col, covariate_col]).size().unstack(fill_value=0)
    counts = counts.reindex(index=cluster_order, columns=cov_order, fill_value=0)

    if normalize == "cluster":
        prop = counts.div(counts.sum(axis=1).replace(0, 1), axis=0)
        y_label_default = "Proportion of cluster"
    elif normalize == "covariate":
        prop = counts.div(counts.sum(axis=0).replace(0, 1), axis=1)
        y_label_default = f"Proportion of {covariate_col}"
    else:
        raise ValueError(f"normalize must be 'cluster' or 'covariate', got {normalize!r}")

    x_vals = [_fmt_cluster(c) for c in cluster_order]

    fig = go.Figure()
    for cov in cov_order:
        ys = [prop.loc[c, cov] for c in cluster_order]
        fig.add_trace(go.Bar(
            x=x_vals, y=ys,
            name=str(cov),
            marker=dict(color=palette.get(cov, "#888"),
                        line=dict(width=0.4, color="rgba(255,255,255,0.7)")),
            hovertemplate=(
                f"<b>{cov}</b><br>"
                f"{cluster_col}: %{{x}}<br>"
                "Proportion: %{y:.3f}<extra></extra>"
            ),
        ))

    fig.update_layout(
        barmode="stack", bargap=bargap,
        **_base_layout(width, height,
                       legend_title=legend_title or f"<b>{covariate_col}</b>"),
    )
    fig.update_xaxes(
        type="category",
        title_text=x_title or cluster_col,
        title_font=dict(size=FONT_AXIS_TITLE, family=FONT_FAMILY),
        tickfont=dict(size=FONT_TICK, family=FONT_FAMILY),
        tickangle=tick_rotation,
        showgrid=False,
    )
    fig.update_yaxes(
        title_text=y_title or y_label_default,
        title_font=dict(size=FONT_AXIS_TITLE, family=FONT_FAMILY),
        tickfont=dict(size=FONT_TICK, family=FONT_FAMILY),
        range=[0, 1.001], dtick=0.2,
        gridcolor="rgba(200,200,200,0.3)", gridwidth=0.5,
        zeroline=False,
    )

    if show_counts and normalize == "cluster":
        for i, c in enumerate(cluster_order):
            n = int(counts.loc[c].sum())
            fig.add_annotation(
                x=x_vals[i], y=1.015, text=f"n={n}",
                showarrow=False,
                font=dict(size=FONT_TICK - 2, color="#666", family=FONT_FAMILY),
                yref="y",
            )
    fig.update_layout(showlegend=show_legend)
    return fig


# =======================================================================
# UMAP w/ cluster median anchors — badge-style markers (manuscript figure panel)
# =======================================================================
def umap_with_medians(df,
                      x_col=None, y_col=None,
                      cluster_col=None, palette=None,
                      bg_marker_size=4, bg_opacity=0.45,
                      median_size=22, median_text_size=11,
                      halo_color="rgba(255,255,255,0.92)",
                      halo_width=2.5,
                      auto_text_contrast=True,
                      width=950, height=750,
                      show_legend=True,
                      legend_title=None,
                      cluster_label_prefix=None,
                      draw=None, seed=None):
    """
    UMAP scatter w/ a low-opacity background per cluster, overlaid with
    'badge' median markers — filled circles in the cluster color with the
    cluster ID printed inside (white text, picked for contrast on pale fills).

    Premium-figure aesthetic — replaces diamond markers + side text labels
    + black borders.

    Defaults to the SHAPE embedding. Pass x_col='niche_umap_1' /
    y_col='niche_umap_2' (and a niche cluster_col) for the niche version.
    """
    if x_col is None: x_col = "umap_1"
    if y_col is None: y_col = "umap_2"
    is_niche = x_col.startswith("niche_") or y_col.startswith("niche_")
    if cluster_col is None:
        cluster_col = (f"niche_leiden_r{NICHE_RESOLUTION}" if is_niche
                       else f"leiden_r{SHAPE_RESOLUTION}")
    if palette is None:
        palette = NICHE_CLUSTER_CMAP if is_niche else SHAPE_CLUSTER_CMAP
    if cluster_label_prefix is None:
        cluster_label_prefix = "N" if is_niche else "C"
    if legend_title is None:
        legend_title = f"<b>{'Niche' if is_niche else 'Shape'} cluster</b>"

    d = df.dropna(subset=[x_col, y_col, cluster_col])
    clusters = sorted(d[cluster_col].astype(int).unique())

    fig = go.Figure()

    # ---- background scatter (dimmed; intermixed unless draw="stacked") ----
    d = d.assign(_cluster=d[cluster_col].astype(int))
    counts = d["_cluster"].value_counts()
    _grouped_scatter(
        fig, d, x_col, y_col, "_cluster", clusters, palette,
        names={c: f"{cluster_label_prefix}{int(c)} (n={int(counts[c]):,})" for c in clusters},
        hover={c: f"<b>{cluster_label_prefix}{int(c)}</b>" for c in clusters},
        legendgroups={c: f"c{c}" for c in clusters},
        marker_size=bg_marker_size, opacity=bg_opacity,
        draw=draw, seed=seed, biggest_first=False)

    # ---- median anchor badges (per cluster, drawn on top) ----
    for c in clusters:
        sub = d[d[cluster_col].astype(int) == c]
        if sub.empty:
            continue
        cx = sub[x_col].median()
        cy = sub[y_col].median()
        fill_hex = palette.get(int(c), "#888888")
        text_color = (_readable_text_color(fill_hex)
                      if auto_text_contrast else "#FFFFFF")
        fig.add_trace(go.Scatter(
            x=[cx], y=[cy],
            mode="markers+text",
            marker=dict(
                size=median_size,
                color=fill_hex,
                line=dict(color=halo_color, width=halo_width),
                symbol="circle",
            ),
            text=[str(int(c))],
            textposition="middle center",
            textfont=dict(size=median_text_size, color=text_color,
                          family=FONT_FAMILY),
            name=f"median {cluster_label_prefix}{int(c)}",
            legendgroup=f"c{c}",
            showlegend=False,
            hovertemplate=(
                f"<b>median {cluster_label_prefix}{int(c)}</b><br>"
                "%{x:.2f}, %{y:.2f}<extra></extra>"
            ),
        ))

    fig.update_layout(**_base_layout(width, height,
                                      legend_title=legend_title,
                                      show_legend=show_legend))
    _apply_umap_axes(fig,
                     x_title="Niche UMAP 1" if is_niche else "UMAP 1",
                     y_title="Niche UMAP 2" if is_niche else "UMAP 2")
    fig.update_layout(showlegend=show_legend)
    return fig


print("Plot functions defined:",
      "umap_by_{sample,pattern,donor,tissue,timepoint,cluster}, ",
      "umap_by_feature, umap_with_medians, ",
      "violin_panel, composition_bar, enrichment_heatmap")


---

# Niche UMAP — all the coverings (mirrors notebook 03)

Same functions, called with `x_col="niche_umap_1"` / `y_col="niche_umap_2"`.
Cells with no adjacency are dropped automatically.


### Niche UMAP — by pattern

In [ ]:
fig = umap_by_pattern(df, x_col="niche_umap_1", y_col="niche_umap_2")
fig.show()
# save_fig(fig, "niche_umap_by_pattern")  # native resolution; scale in Illustrator

### Niche UMAP — by sample

In [ ]:
fig = umap_by_sample(df, x_col="niche_umap_1", y_col="niche_umap_2")
fig.show()
save_fig(fig, "niche_umap_by_sample")  # Fig. 7D; native resolution; scale in Illustrator

In [ ]:
fig = umap_by_sample(df, x_col="umap_1", y_col="umap_2")
fig.show()
save_fig(fig, "shape_umap_by_sample")  # Fig. 7C


### Niche UMAP — by tissue

In [ ]:
fig = umap_by_tissue(df, x_col="niche_umap_1", y_col="niche_umap_2")
fig.show()
save_fig(fig, "niche_umap_by_tissue")  # Fig. 7B; native resolution; scale in Illustrator

In [ ]:
fig = umap_by_tissue(df, x_col="umap_1", y_col="umap_2")
fig.show()
save_fig(fig, "shape_umap_by_tissue")  # Fig. 7A


### Niche UMAP — by timepoint

In [ ]:
fig = umap_by_timepoint(df, x_col="niche_umap_1", y_col="niche_umap_2")
fig.show()
# save_fig(fig, "niche_umap_by_timepoint")  # native resolution; scale in Illustrator

In [ ]:
fig = umap_by_timepoint(df, x_col="umap_1", y_col="umap_2")
fig.show()

## Cross-embedding views

**Niche UMAP colored by shape cluster** — does a niche cluster correspond to
a single shape cluster, or is it a mixture? (The niche colors a region by
"who lives here"; the shape colors answer "are they all the same morphology?")

**Shape UMAP colored by niche cluster** — the inverse view: does a shape
cluster reside in one tissue niche or spread across several?


### Niche UMAP colored by SHAPE cluster

In [ ]:
# Niche UMAP colored by SHAPE cluster
fig = umap_by_cluster(
    df, x_col="niche_umap_1", y_col="niche_umap_2",
    cluster_col=f"leiden_r{SHAPE_RESOLUTION}",
    palette=SHAPE_CLUSTER_CMAP,
    legend_title="<b>Shape cluster</b>",
)
fig.show()
save_fig(fig, "niche_umap_by_shape_cluster")  # native resolution; scale in Illustrator


### Shape UMAP colored by NICHE cluster

In [ ]:
# Shape UMAP colored by NICHE cluster
fig = umap_by_cluster(
    df, x_col="umap_1", y_col="umap_2",
    cluster_col=f"niche_leiden_r{NICHE_RESOLUTION}",
    palette=NICHE_CLUSTER_CMAP,
    legend_title="<b>Niche cluster</b>",
)
fig.show()
save_fig(fig, "shape_umap_by_niche_cluster")  # native resolution; scale in Illustrator


---

## Figure UMAPs — cluster median anchors (badge style)

Premium-aesthetic UMAP with overlaid median markers: filled circles in the
cluster color, cluster ID printed inside (white or dark text depending on
the fill's luminance — auto), no harsh black border, soft white halo for
print-contrast.

Use these as the main figure UMAP panels. The mesh gallery cells below
render the corresponding cluster representatives — drop the gallery
adjacent in Illustrator (or InDesign / Affinity) for the canonical
"UMAP + per-cluster meshes" figure layout.


### Shape UMAP w/ median anchors

In [ ]:
# Shape UMAP w/ median anchors
fig = umap_with_medians(df)
fig.show()
save_fig(fig, "shape_umap_medians")  # native resolution; scale in Illustrator


### Niche UMAP w/ median anchors

In [ ]:
# Niche UMAP w/ median anchors
fig = umap_with_medians(df,
                        x_col="niche_umap_1", y_col="niche_umap_2")
fig.show()
save_fig(fig, "niche_umap_medians")  # native resolution; scale in Illustrator


## Feature-colored UMAP

Continuous colormap. Pass any numeric column from `df` — shape features,
PCs, niche PCs, QC scores, etc. Use `log_transform=True` for heavy-tailed
features like `area` or `surface_area`.


### Fig. 6C -- shape features on the shape UMAP (rank-transformed)

In [ ]:
plot_feature = [
    "area",
    "z_span",
    "apical_basal_area_ratio",
    "sphericity",
    "xy_elongation"
]
for item in plot_feature:
    fig = umap_by_feature(df, item, transform="rank",
                x_col="umap_1", y_col="umap_2",) # colorscale='RdBu_r')
    fig.show()
    save_fig(fig, f"shape_umap_{item}_rank") 

## Violin — single feature across a categorical grouping

One call = one panel. Change `feature` and/or `group_col`. Useful groupings:

- `group_col="leiden_r1.0"` → shape-cluster morphology fingerprint
- `group_col="niche_leiden_r1.0"` → do niche clusters differ morphologically?
- `group_col="pattern"` → patterned vs unpatterned comparison
- `group_col="sample_short"` → inter-sample variance (use `tick_rotation=40`)
- `group_col="timepoint"` → developmental trajectory (use `show_median_line=True`)


### Example 2 — aspect_ratio across niche clusters

In [ ]:
# Aspect ratio across niche clusters -- does niche predict morphology?
fig = violin_panel(df, feature="aspect_ratio",
             group_col=f"niche_leiden_r{NICHE_RESOLUTION}",
             palette=NICHE_CLUSTER_CMAP,
             x_title="Niche cluster")
fig.show()
# save_fig(fig, "violin_aspect_ratio_niche_clusters")  # native resolution; scale in Illustrator


### Example 3 — true_sphericity across epithelial patterns

In [ ]:
# Feature across patterns -- back/front/dorsal/unpatterned/ridged/whorl comparison
df_ = _ensure_sample_short(df)
fig = violin_panel(df_, feature="true_sphericity",
             group_col="pattern",
             palette=PATTERN_PALETTE,
             group_order=PATTERN_ORDER,
             x_title="Epithelial pattern")
fig.show()
# save_fig(fig, "violin_true_sphericity_patterns")  # native resolution; scale in Illustrator


### Example 4 — surface_area across samples (log-scale)

In [ ]:
# Across samples -- inter-sample variance (rotate x labels for readability)
df_ = _ensure_sample_short(df)
fig = violin_panel(df_, feature="surface_area",
             group_col="sample_short",
             palette=SAMPLE_PALETTE,
             group_order=[SHORT[s] for s in STACK_ORDER if SHORT[s] in df_["sample_short"].unique()],
             y_transform="log1p",
             width=1100, height=540,
             tick_rotation=35,
             x_title="Sample")
fig.show()
# save_fig(fig, "violin_surface_area_samples")  # native resolution; scale in Illustrator


### Example 5 — area across timepoints (median trend)

In [ ]:
# Across timepoints with median trend line
df_ = _ensure_sample_short(df)
fig = violin_panel(df_, feature="area",
             group_col="timepoint",
             palette=TIMEPOINT_PALETTE,
             group_order=[t for t in TIMEPOINT_ORDER if t in df_["timepoint"].unique()],
             y_transform="log1p",
             show_median_line=True,
             x_title="Timepoint")
fig.show()
# save_fig(fig, "violin_area_timepoints")  # native resolution; scale in Illustrator


---

## Compositional stacked bars — cluster x covariate

**Why it matters.** If a cluster is dominated by one donor/sample, the
clustering is probably batch-driven rather than biological. Stacked bars
decompose each cluster into its covariate composition: a healthy cluster
should be a mix; a batch-effect cluster will be ~monochromatic.

`composition_bar(df, cluster_col, covariate_col, palette, ...)` is fully
parameterized. `normalize="cluster"` (default) makes each bar sum to 1 —
the covariate mix within each cluster. `normalize="covariate"` flips it —
how each covariate spreads across clusters.


### Niche cluster x tissue

In [ ]:
# Niche clusters x tissue (back / front / dorsal / volar)
fig = composition_bar(df,
                cluster_col=f"niche_leiden_r{NICHE_RESOLUTION}",
                covariate_col="tissue",
                palette=TISSUE_PALETTE, cov_order=TISSUE_ORDER,
                x_title="Niche cluster")
fig.show()
# save_fig(fig, "composition_niche_by_tissue")  # native resolution; scale in Illustrator


### Niche cluster x timepoint

In [ ]:
# Niche clusters x timepoint
fig = composition_bar(df,
                cluster_col=f"niche_leiden_r{NICHE_RESOLUTION}",
                covariate_col="timepoint",
                palette=TIMEPOINT_PALETTE,
                cov_order=[t for t in TIMEPOINT_ORDER if t in df["timepoint"].unique()],
                x_title="Niche cluster")
fig.show()
# save_fig(fig, "composition_niche_by_timepoint")  # native resolution; scale in Illustrator


## Enrichment heatmaps — log2(observed/expected)

Two common comparisons:

- **Pattern × niche cluster** — which niches are enriched in which epithelial patterns?
- **Shape cluster × niche cluster** — does morphology co-localize with tissue niche?

Red = enriched (observed > expected), blue = depleted. Clipped at ±3.


### Pattern × niche-cluster enrichment

In [ ]:
fig = enrichment_heatmap(
    df, row_col="pattern", col_col=f"niche_leiden_r{NICHE_RESOLUTION}",
    row_order=PATTERN_ORDER,
    col_label_prefix="N", width=1000, height=380,
    title=f"Pattern x Niche cluster enrichment (niche config = {NICHE_CONFIG})",
    x_title="Niche cluster", y_title="Epithelial pattern",
)
fig.show()
# save_fig(fig, "enrichment_pattern_by_niche")  # native resolution; scale in Illustrator


### Shape-cluster × niche-cluster enrichment

In [ ]:
fig = enrichment_heatmap(
    df, row_col=f"leiden_r{SHAPE_RESOLUTION}",
    col_col=f"niche_leiden_r{NICHE_RESOLUTION}",
    row_label_prefix="S", col_label_prefix="N",
    width=1000, height=max(400, 32 * df[f"leiden_r{SHAPE_RESOLUTION}"].nunique()),
    title=f"Shape x Niche enrichment ({NICHE_CONFIG}, shape_res={SHAPE_RESOLUTION}, niche_res={NICHE_RESOLUTION})",
    x_title="Niche cluster", y_title="Shape cluster",
)
fig.show()
# save_fig(fig, "enrichment_shape_by_niche")  # native resolution; scale in Illustrator


### Reference: Pattern × shape-cluster enrichment

In [ ]:
# Pattern x SHAPE cluster (reference — pure shape story, no niche involved)
fig = enrichment_heatmap(
    df, row_col="pattern", col_col=f"leiden_r{SHAPE_RESOLUTION}",
    row_order=PATTERN_ORDER,
    col_label_prefix="S", width=1000, height=380,
    title=f"Pattern x Shape cluster enrichment (res={SHAPE_RESOLUTION})",
    x_title="Shape cluster", y_title="Epithelial pattern",
)
fig.show()
# save_fig(fig, "enrichment_pattern_by_shape")  # native resolution; scale in Illustrator


## Niche mesh gallery

Renders representative cells for each **niche** cluster using PyVista.
Representatives were pre-selected by notebook 02 as the 10 cells closest to
each niche cluster's median in niche-PC space, and saved to
`run/niche_{NICHE_CONFIG}/leiden_res_{res}/representative_cells.csv`.

Sizing follows the `RELATIVE_SIZES` knob in section 2:

- `RELATIVE_SIZES = True` (default) -- single shared divisor across the gallery,
  so the largest representative fills its panel and smaller cells render
  proportionally smaller. This preserves the real cell-size differences
  between clusters.
- `RELATIVE_SIZES = False` -- per-cell normalization, every cell rescaled to
  fill its own panel (better for comparing pure shape).

**Main figure**: 1 representative per cluster (closest-to-median).
**Supplementary**: 10 per cluster (full grid).


In [ ]:
import matplotlib.pyplot as plt

try:
    import pyvista as pv
    import trimesh
    pv.OFF_SCREEN = True
    HAS_PYVISTA = True
except ImportError:
    HAS_PYVISTA = False
    print("PyVista/trimesh not available -- gallery rendering disabled")


_CAM_CFGS = {
    "iso": dict(pos=(2.8, 2.2, 1.6), up=(0, 0, 1)),
    "xz":  dict(pos=(0.0, -3.2, 0.0), up=(0, 0, 1)),
    "xy":  dict(pos=(0.0, 0.0, 3.2), up=(0, 1, 0)),
    "yz":  dict(pos=(3.2, 0.0, 0.0), up=(0, 0, 1)),
}


def load_cell_mesh(mesh_path):
    """Load a cell .obj, center at origin, return (verts_xyz, faces, rms_um).

    Vertices are NOT normalized -- the caller picks the divisor so we can
    share one divisor across the whole gallery (relative sizes) or use a
    per-cell divisor (every cell fills its panel). Returns None on failure
    or for degenerate meshes.
    """
    try:
        tm = trimesh.load(mesh_path, file_type="obj", process=False)
    except Exception:
        return None
    verts = np.array(tm.vertices, dtype=float)
    faces = np.array(tm.faces, dtype=int)
    if len(verts) == 0 or len(faces) == 0:
        return None
    verts = verts[:, [2, 1, 0]]                   # ZYX -> XYZ (matches NB08 history)
    verts -= verts.mean(axis=0)
    rms = float(np.sqrt((verts ** 2).sum(axis=1).mean()))
    if rms <= 0:
        return None
    return verts, faces, rms


def render_normalized(verts_norm, faces, color="#4ECDC4",
                      camera_position="iso",
                      window_size=(800, 800),
                      background="#FFFFFF",
                      smooth_iter=30):
    """Render a pre-normalized mesh (vertices already divided by chosen divisor)."""
    pv_faces = np.hstack([np.full((len(faces), 1), 3, dtype=np.int64),
                          faces.astype(np.int64)]).ravel()
    mesh = pv.PolyData(verts_norm, pv_faces)
    if smooth_iter > 0:
        mesh = mesh.smooth(n_iter=smooth_iter, relaxation_factor=0.08)

    cfg = _CAM_CFGS.get(camera_position, _CAM_CFGS["iso"])
    pl = pv.Plotter(off_screen=True, window_size=window_size)
    pl.set_background(background)
    pl.add_mesh(mesh, color=color, smooth_shading=True,
                specular=0.45, specular_power=25,
                ambient=0.15, diffuse=0.85)
    pl.camera.position = cfg["pos"]
    pl.camera.focal_point = (0, 0, 0)
    pl.camera.up = cfg["up"]
    pl.remove_all_lights()
    pl.add_light(pv.Light(position=(3.5, 3.0, 5.0), intensity=0.80))
    pl.add_light(pv.Light(position=(-2.0, -2.0, 2.0), intensity=0.28))
    pl.add_light(pv.Light(position=(0.0, 2.0, -2.5), intensity=0.18))
    img = pl.screenshot(None, return_img=True)
    pl.close()
    return img


def render_cell(mesh_path, color="#4ECDC4",
                camera_position="iso",
                window_size=(800, 800),
                background="#FFFFFF",
                smooth_iter=30):
    """Per-cell normalized render (every cell fills its panel).

    Convenience wrapper for ad-hoc single-cell renders. Galleries should
    use ``render_reps_with_relative_sizes()`` so sizes are comparable
    across the whole gallery.
    """
    loaded = load_cell_mesh(mesh_path)
    if loaded is None:
        return None
    verts, faces, rms = loaded
    return render_normalized(verts / (rms * 1.8), faces, color=color,
                             camera_position=camera_position,
                             window_size=window_size, background=background,
                             smooth_iter=smooth_iter)


def render_reps_with_relative_sizes(rep_df, cmap,
                                    relative_sizes=True,
                                    camera_position="iso",
                                    window_size=(800, 800),
                                    background="#FFFFFF",
                                    smooth_iter=30,
                                    progress_every=20,
                                    progress_label=""):
    """Two-pass renderer used by every NB08 gallery cell.

    Pass 1: load all meshes referenced by ``rep_df`` (centering, RMS).
    Pass 2: render each mesh with a SHARED divisor (largest cell fills
            its panel, others render smaller) when ``relative_sizes`` is
            True, or a PER-CELL divisor (every cell fills its panel)
            when False.

    Required columns in ``rep_df``: ``cluster``, ``role`` (e.g. ``rep_0``),
    ``mesh_path``. Optional: ``mesh_exists``.

    Returns ``(thumbnails, meta)`` where ``thumbnails`` is the
    ``{(cluster_id, rep_idx): RGBA}`` dict consumed by ``build_gallery``.
    """
    # --- Pass 1: load + RMS ---
    loaded = {}
    n_target = 0
    for _, row in rep_df.iterrows():
        c = int(row["cluster"])
        role = str(row.get("role", ""))
        rep_idx = int(role.replace("rep_", "")) if role.startswith("rep_") else 0
        mp = str(row.get("mesh_path", ""))
        if not mp or not os.path.exists(mp):
            continue
        n_target += 1
        built = load_cell_mesh(mp)
        if built is None:
            continue
        loaded[(c, rep_idx)] = built

    if not loaded:
        return {}, {"mode": "none", "n_loaded": 0, "n_target": n_target,
                    "mode_desc": "no meshes loaded"}

    rms_values = [d[2] for d in loaded.values()]
    global_rms = max(rms_values)
    smallest_rms = min(rms_values)

    if relative_sizes:
        shared_divisor = global_rms * 1.8
        def divisor(_rms):
            return shared_divisor
        mode_desc = (f"relative sizes (shared divisor; largest RMS "
                     f"= {global_rms:.2f} um, smallest = {smallest_rms:.2f} um)")
    else:
        def divisor(rms):
            return rms * 1.8
        mode_desc = "per-cell (every cell rescaled to fill its panel)"

    # --- Pass 2: render with chosen divisor ---
    thumbnails = {}
    n_done = 0
    for (c, rep_idx), (verts, faces, rms) in loaded.items():
        color = cmap.get(c, "#888888")
        try:
            img = render_normalized(
                verts / divisor(rms), faces, color=color,
                camera_position=camera_position,
                window_size=window_size, background=background,
                smooth_iter=smooth_iter,
            )
        except Exception as e:
            print(f"  {progress_label}c={c} rep_{rep_idx}: render error: {e}")
            continue
        if img is not None:
            thumbnails[(c, rep_idx)] = img
        n_done += 1
        if progress_every and n_done % progress_every == 0:
            print(f"  Rendered {n_done} / {len(loaded)} ...")

    meta = {
        "mode": "relative" if relative_sizes else "per_cell",
        "mode_desc": mode_desc,
        "n_loaded": len(loaded),
        "n_target": n_target,
        "global_rms_um": global_rms,
        "smallest_rms_um": smallest_rms,
    }
    return thumbnails, meta


def build_gallery(thumbnails, cluster_ids, cmap,
                  n_cols_per_cluster=None,
                  orientation="vertical",
                  cell_size=2.5, save_path=None, dpi=600,
                  cluster_label_prefix="N",
                  suptitle=None):
    """
    thumbnails: {(cluster_id, rep_index): RGBA}
    orientation: 'vertical' (rows=clusters) or 'horizontal' (cols=clusters)
    suptitle:    optional figure title (e.g. relative-size mode description).
    """
    if n_cols_per_cluster is None:
        n_cols_per_cluster = max((idx for (_, idx) in thumbnails.keys()),
                                 default=0) + 1

    n_clusters = len(cluster_ids)
    if orientation == "vertical":
        n_rows, n_cols = n_clusters, n_cols_per_cluster
    else:
        n_rows, n_cols = n_cols_per_cluster, n_clusters

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(cell_size * n_cols, cell_size * n_rows),
                             facecolor="white")
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    if n_cols == 1:
        axes = axes.reshape(-1, 1)

    for ax in axes.flat:
        ax.set_facecolor("white")
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_linestyle(":"); sp.set_linewidth(0.7); sp.set_color("#BBBBBB")

    for ci, c in enumerate(cluster_ids):
        color_hex = cmap.get(c, "#888888")
        for ri in range(n_cols_per_cluster):
            if orientation == "vertical":
                row_idx, col_idx = ci, ri
            else:
                row_idx, col_idx = ri, ci
            ax = axes[row_idx, col_idx]
            key = (c, ri)
            if key in thumbnails:
                ax.imshow(thumbnails[key])
            else:
                ax.text(0.5, 0.5, "--", ha="center", va="center",
                        fontsize=9, color="#999", transform=ax.transAxes)
            if orientation == "vertical" and ri == 0:
                ax.set_ylabel(f"{cluster_label_prefix}{c}",
                              fontsize=10, color=color_hex,
                              fontweight="bold", labelpad=4,
                              fontfamily=FONT_FAMILY)
            elif orientation == "horizontal" and ri == 0:
                ax.set_title(f"{cluster_label_prefix}{c}",
                             fontsize=10, color=color_hex,
                             fontweight="bold", pad=4,
                             fontfamily=FONT_FAMILY)

    if suptitle:
        fig.suptitle(suptitle, y=1.0, fontsize=10, color="#444",
                     family=FONT_FAMILY)
    plt.tight_layout(pad=0.3)
    if save_path:
        # TIFF gets LZW compression (Illustrator + Photoshop read it transparently)
        sp = str(save_path)
        kw = dict(dpi=dpi, bbox_inches="tight", facecolor="white")
        if sp.lower().endswith((".tif", ".tiff")):
            kw["pil_kwargs"] = {"compression": "tiff_lzw"}
        plt.savefig(sp, **kw)
        print(f"Saved: {sp}")
    plt.show()
    return fig


print("render_cell() + render_reps_with_relative_sizes() + build_gallery() defined"
      if HAS_PYVISTA else "Gallery fns defined (PyVista unavailable)")


### Main — 1 median per niche cluster

In [ ]:
# Main-figure gallery: 1 representative per niche cluster (single row)
if HAS_PYVISTA:
    rep_csv = os.path.join(
        MORPHO_OUTPUT_DIR,
        f"niche_{NICHE_CONFIG}",
        f"leiden_res_{NICHE_RESOLUTION}",
        "representative_cells.csv",
    )
    if not os.path.exists(rep_csv):
        print(f"No niche representative CSV at {rep_csv}")
    else:
        rep_df = pd.read_csv(rep_csv)
        rep_df = rep_df[rep_df.get("mesh_exists", True) == True]
        # closest-to-median = rep_0
        rep0 = rep_df[rep_df["role"] == "rep_0"].copy()
        niche_clusters = sorted(rep0["cluster"].astype(int).unique())

        median_thumbs, meta = render_reps_with_relative_sizes(
            rep0, NICHE_CLUSTER_CMAP,
            relative_sizes=RELATIVE_SIZES,
            camera_position="iso",
            progress_every=0,
            progress_label="N",
        )
        print(f"Rendered {len(median_thumbs)} niche median cells "
              f"({meta['mode_desc']})")

        # Persist individual median renders so they can be reused outside the gallery.
        individual_dir = os.path.join(MORPHO_OUTPUT_DIR,
                                      f"niche_{NICHE_CONFIG}_medians")
        os.makedirs(individual_dir, exist_ok=True)
        for (c, rep_idx), img in median_thumbs.items():
            from PIL import Image
            Image.fromarray(img).save(
                os.path.join(individual_dir, f"median_N{c}.tif"),
                format="TIFF", compression="tiff_lzw")

        if median_thumbs:
            size_tag = "relative" if RELATIVE_SIZES else "per_cell"
            out = os.path.join(
                MORPHO_OUTPUT_DIR,
                f"niche_gallery_main_{NICHE_CONFIG}_{size_tag}.tif",
            )
            build_gallery(median_thumbs, niche_clusters,
                          NICHE_CLUSTER_CMAP,
                          n_cols_per_cluster=1,
                          orientation="horizontal",
                          cell_size=3.0,
                          cluster_label_prefix="N",
                          suptitle=f"Niche cluster medians -- {meta['mode_desc']}",
                          save_path=out, dpi=600)
            print(f"Individual renders: {individual_dir}/")
else:
    print("PyVista not available -- skip gallery")


### Supplementary — 10 per niche cluster

In [ ]:
# Supplementary gallery: 10 representatives per niche cluster (full grid)
if HAS_PYVISTA:
    rep_csv = os.path.join(
        MORPHO_OUTPUT_DIR,
        f"niche_{NICHE_CONFIG}",
        f"leiden_res_{NICHE_RESOLUTION}",
        "representative_cells.csv",
    )
    if not os.path.exists(rep_csv):
        print(f"No niche representative CSV at {rep_csv}")
    else:
        rep_df = pd.read_csv(rep_csv)
        rep_df = rep_df[rep_df.get("mesh_exists", True) == True]
        niche_clusters = sorted(rep_df["cluster"].astype(int).unique())

        supp_thumbs, meta = render_reps_with_relative_sizes(
            rep_df, NICHE_CLUSTER_CMAP,
            relative_sizes=RELATIVE_SIZES,
            camera_position="iso",
            progress_every=20,
            progress_label="N",
        )
        print(f"Rendered {len(supp_thumbs)} thumbnails total "
              f"({meta['mode_desc']})")

        if supp_thumbs:
            size_tag = "relative" if RELATIVE_SIZES else "per_cell"
            out = os.path.join(
                MORPHO_OUTPUT_DIR,
                f"niche_gallery_supp_{NICHE_CONFIG}_res{NICHE_RESOLUTION}_{size_tag}.tif",
            )
            build_gallery(supp_thumbs, niche_clusters, NICHE_CLUSTER_CMAP,
                          orientation="horizontal", cell_size=2.5,
                          cluster_label_prefix="N",
                          suptitle=f"Niche reps (10 per cluster) -- {meta['mode_desc']}",
                          save_path=out, dpi=600)
gc.collect()


## Shape mesh gallery

Same render pipeline as the niche gallery, pointed at the shape
representative CSVs at
`run/leiden_res_{SHAPE_RESOLUTION}/representative_cells.csv`
(generated by notebook 01). Honors the same `RELATIVE_SIZES` knob from
section 2.

**Main**: 1 representative per shape cluster (closest-to-median).
**Supplementary**: 10 per cluster -- full grid.


### Shape main — 1 median per shape cluster

In [ ]:
# Shape main gallery: 1 representative per cluster (single row)
if HAS_PYVISTA:
    rep_csv = os.path.join(MORPHO_OUTPUT_DIR,
                           f"leiden_res_{SHAPE_RESOLUTION}",
                           "representative_cells.csv")
    if not os.path.exists(rep_csv):
        print(f"No shape representative CSV at {rep_csv}")
    else:
        rep_df = pd.read_csv(rep_csv)
        rep_df = rep_df[rep_df.get("mesh_exists", True) == True]
        rep0 = rep_df[rep_df["role"] == "rep_0"].copy()
        shape_clusters = sorted(rep0["cluster"].astype(int).unique())

        median_thumbs, meta = render_reps_with_relative_sizes(
            rep0, SHAPE_CLUSTER_CMAP,
            relative_sizes=RELATIVE_SIZES,
            camera_position="iso",
            progress_every=0,
            progress_label="C",
        )
        print(f"Rendered {len(median_thumbs)} shape median cells "
              f"({meta['mode_desc']})")

        individual_dir = os.path.join(MORPHO_OUTPUT_DIR, "shape_medians")
        os.makedirs(individual_dir, exist_ok=True)
        for (c, rep_idx), img in median_thumbs.items():
            from PIL import Image
            Image.fromarray(img).save(
                os.path.join(individual_dir, f"median_C{c}.tif"),
                format="TIFF", compression="tiff_lzw")

        if median_thumbs:
            size_tag = "relative" if RELATIVE_SIZES else "per_cell"
            out = os.path.join(MORPHO_OUTPUT_DIR,
                               f"shape_gallery_main_{size_tag}.tif")
            build_gallery(median_thumbs, shape_clusters,
                          SHAPE_CLUSTER_CMAP,
                          n_cols_per_cluster=1,
                          orientation="horizontal",
                          cell_size=3.0,
                          cluster_label_prefix="C",
                          suptitle=f"Shape cluster medians -- {meta['mode_desc']}",
                          save_path=out, dpi=600)
            print(f"Individual renders: {individual_dir}/")
else:
    print("PyVista not available -- skip gallery")


### Shape supplementary — 10 per shape cluster

In [ ]:
# Shape supplementary gallery: 10 representatives per cluster (full grid)
if HAS_PYVISTA:
    rep_csv = os.path.join(MORPHO_OUTPUT_DIR,
                           f"leiden_res_{SHAPE_RESOLUTION}",
                           "representative_cells.csv")
    if not os.path.exists(rep_csv):
        print(f"No shape representative CSV at {rep_csv}")
    else:
        rep_df = pd.read_csv(rep_csv)
        rep_df = rep_df[rep_df.get("mesh_exists", True) == True]
        shape_clusters = sorted(rep_df["cluster"].astype(int).unique())

        supp_thumbs, meta = render_reps_with_relative_sizes(
            rep_df, SHAPE_CLUSTER_CMAP,
            relative_sizes=RELATIVE_SIZES,
            camera_position="iso",
            progress_every=20,
            progress_label="C",
        )
        print(f"Rendered {len(supp_thumbs)} thumbnails total "
              f"({meta['mode_desc']})")

        if supp_thumbs:
            size_tag = "relative" if RELATIVE_SIZES else "per_cell"
            out = os.path.join(
                MORPHO_OUTPUT_DIR,
                f"shape_gallery_supp_res{SHAPE_RESOLUTION}_{size_tag}.tif",
            )
            build_gallery(supp_thumbs, shape_clusters, SHAPE_CLUSTER_CMAP,
                          orientation="horizontal", cell_size=2.5,
                          cluster_label_prefix="C",
                          suptitle=f"Shape reps (10 per cluster) -- {meta['mode_desc']}",
                          save_path=out, dpi=600)
gc.collect()


---

## Saving figures for the paper

`save_fig(fig, name)` writes both an Illustrator-editable **SVG** (vector) and a
**high-DPI PNG** sidecar (300 dpi). **Default = native resolution** (whatever
width/height the plot function was tuned for, typically 900x680 px). SVG is
vector, so you scale it freely in Illustrator without quality loss.

If you need exact physical sizing for an A4 layout, pass the optional `mm=`
argument:

Presets:

| Preset             | Size (mm) | Use case                                 |
| ------------------ | --------- | ---------------------------------------- |
| `A4_PORTRAIT_MM`   | 210 x 297 | Full A4 page, portrait                   |
| `A4_LANDSCAPE_MM`  | 297 x 210 | Full A4 page, landscape                  |
| `PANEL_FULL_MM`    | 180 x 130 | Full-width manuscript panel              |
| `PANEL_HALF_MM`    | 86 x 70   | Half-width / single-column panel         |

Usage:

```python
fig = umap_by_pattern(df)
fig.show()
save_fig(fig, "umap_by_pattern")
# Native resolution (matches the figure function defaults).
# Scale freely in Illustrator -- SVG is vector, no quality loss.
# -> run/figures/umap_by_pattern.svg + run/figures/umap_by_pattern.png
```

Custom pixel size:

```python
save_fig(fig, "custom", width_px=900, height_px=600)
```

**Setup**: requires Plotly's static-image backend.

```bash
pip install -U kaleido
```


## napari launcher — multi-config viewer

Loads one target stack in napari with:

- a cluster-colored labels layer **per niche config** (each config as a
  separate togglable layer — toggle in the layer list to compare spatial
  domain detection across configs)
- a shape-cluster layer (same stack)
- per-cluster orientation vectors (positive-Z sign flip)

Interactive exploration only — **not for export**. Use this while writing
the figure caption to decide which config's spatial domains read most
clearly.


In [ ]:
# ---- USER KNOBS ----
# TARGET_STACK = (
#     df.loc[df[f"niche_leiden_r{NICHE_RESOLUTION}"].notna(), "stack_name"]
#       .value_counts().idxmax()
# )
# TARGET_STACK   = '(13 wks) D3L ventral ridges x40-Smooth05_Resampled.tif' # DONE
# TARGET_STACK   = '(13 wks) Front skin x40-Smooth05_Resampled.tif' # DONE
TARGET_STACK = '(14 wks) digit 5 unpatterned skin x40 stack-Smooth05_Resampled.tif' # REVISIT ADJACENCY ISOMETRIC
# TARGET_STACK   = '(14 wks) ventral whorl x40-Smooth05_Resampled.tif' # DONE

# --- Physical calibration -------------------------------------------------
# Voxel size in micrometers, (z, y, x). Everything in the viewer is rendered
# in PHYSICAL units: layers carry `scale=`, features stay in um (no /SPACING).
SPACING      = 0.21                          # um per voxel (isotropic, resampled)
VOXEL_SIZE   = (SPACING, SPACING, SPACING)   # (z, y, x) um
SOURCE_VOXEL = VOXEL_SIZE                    # set separately if the raw stack
                                             # was NOT resampled the same way
SCALE_BAR_UNIT = "um"

# Sizes below are now in MICROMETERS, not voxels.
VECTOR_LEN      = 10.0   # orientation vector length (um)  [was 50 voxels ~= 10.5 um]
VECTOR_WIDTH    = 0.6    # um  [was 3 voxels]
EDGE_WIDTH      = 0.4    # um  [was 2 voxels]
CENTROID_SIZE   = 0.9    # um  [was 4 voxels]
# --------------------


# Which overlays to build (toggle layers still visible via napari layer list)
PLOT_OPTIONS = {
    "source_fluorescence": True,   # original microscopy stack (per channel)
    "raw_segmentation":    True,   # original labels, each cell its own color
    "shape_clusters":      True,   # labels colored by shape leiden
    "niche_configs":       True,   # one layer per niche config
    "orientation_vectors": False,  # per-niche-cluster direction lines
    "adjacency_edges":     True,   # contact graph (reads <stack>_adjacency.csv)
    "cell_centroids":      False,  # points at each centroid
}

# Vector appearance — "line" (no arrowhead, publication-safe) or "arrow"
VECTOR_STYLE = "line"

# Put the centroid at the midpoint of each orientation vector (vs at the start).
# Visually nicer for bidirectional axes (major_axis has arbitrary +/- sign).
CENTROID_AS_MIDPOINT = True

# Which niche configs to render as layers (labels + orientation vectors).
# Default: just the featured config — manuscript-clean. Set to None to plot
# all configs (prior behavior), or a list like ["hop1_w0.5", "hop2_w0.5"]
# for A/B comparison.
NICHE_CONFIGS_TO_PLOT = [NICHE_CONFIG]

# Adjacency options (only used when PLOT_OPTIONS["adjacency_edges"] is True)
EDGE_COLOR_MODE = "white"  # 'cluster' | 'contact' | 'white'
MIN_CONTACT     = 100        # min contact_voxels to include an edge
# =======================================================================

import json as _json

def _hex_to_rgba(h, a=1.0):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) / 255 for i in (0, 2, 4)) + (a,)


state_path = "workspace/stack_segmentation_state.json"
stack_paths = {}
stack_records = {}
if os.path.exists(state_path):
    with open(state_path) as _f:
        _state = _json.load(_f)
    for sn, info in _state.get("stacks", {}).items():
        lp = info.get("labels_path", "")
        if lp and not os.path.isabs(lp):            # recorded relative to the workspace root
            lp = os.path.join("workspace", lp)
        if lp and os.path.exists(lp):
            stack_paths[sn] = lp
            stack_records[sn] = info

if TARGET_STACK not in stack_paths:
    print(f"Stack not found: {TARGET_STACK[:60]}")
    print("Available:", list(stack_paths.keys())[:5])
else:
    import napari
    import tifffile
    from napari.utils.colormaps import DirectLabelColormap

    labels = tifffile.imread(stack_paths[TARGET_STACK])
    max_label = int(labels.max())
    stack_df = df[df["stack_name"] == TARGET_STACK].copy()

    def _cluster_volume_and_cmap(row_col, cmap):
        """Relabel -> cluster IDs (+1 so 0=background). Returns (vol, DirectLabelColormap)."""
        lut = np.zeros(max_label + 1, dtype=np.int32)
        valid = stack_df.dropna(subset=[row_col])
        for _, row in valid.iterrows():
            lid = int(row["label"])
            cid = int(row[row_col])
            if 0 < lid <= max_label:
                lut[lid] = cid + 1
        vol = lut[labels]
        color_dict = {0: (0, 0, 0, 0)}
        for cid in sorted(valid[row_col].astype(int).unique()):
            color_dict[int(cid) + 1] = _hex_to_rgba(cmap.get(int(cid), "#888"))
        return vol, DirectLabelColormap(color_dict=color_dict)

    def _build_adjacency(row_col, cmap, adj_path):
        """Build adjacency edges + centroids for `row_col` coloring.

        Returns (edges_arr, edge_color, points, pt_colors) or None if no edges.
        edges_arr is (N, 2, 3): start point + direction for napari Vectors.

        NOTE: coordinates stay in MICROMETERS (as stored in the feature table).
        Layers get `scale=VOXEL_SIZE`, so napari's world space is um and the
        scale bar reads correctly.
        """
        adj = pd.read_csv(adj_path)
        adj = adj[adj["contact_voxels"] >= MIN_CONTACT].copy()

        centroid_lookup = {}
        for _, row in stack_df.iterrows():
            cent = np.array([row["centroid-0"], row["centroid-1"], row["centroid-2"]])
            centroid_lookup[int(row["label"])] = cent

        valid = stack_df.dropna(subset=[row_col])
        label_to_cluster = dict(
            zip(valid["label"].astype(int), valid[row_col].astype(int))
        )

        edges, edge_sources = [], []
        for _, row in adj.iterrows():
            a, b = int(row["label_a"]), int(row["label_b"])
            if a not in centroid_lookup or b not in centroid_lookup:
                continue
            edges.append([centroid_lookup[a], centroid_lookup[b] - centroid_lookup[a]])
            edge_sources.append(a)
        if not edges:
            return None

        edges_arr = np.array(edges)
        if EDGE_COLOR_MODE == "cluster":
            colors = np.zeros((len(edges_arr), 4), dtype=float)
            for i, src in enumerate(edge_sources):
                cid = label_to_cluster.get(src, -1)
                colors[i] = _hex_to_rgba(cmap.get(int(cid), "#888"), a=0.5)
            edge_color = colors
        elif EDGE_COLOR_MODE == "contact":
            contacts = adj["contact_voxels"].values[:len(edges_arr)]
            norm = np.log1p(contacts.astype(float))
            if norm.max() > norm.min():
                norm = (norm - norm.min()) / (norm.max() - norm.min())
            colors = np.zeros((len(edges_arr), 4), dtype=float)
            colors[:, 0] = norm
            colors[:, 2] = 1.0 - norm
            colors[:, 3] = 0.5
            edge_color = colors
        else:
            edge_color = "white"

        sorted_labels = sorted(centroid_lookup.keys())
        points = np.array([centroid_lookup[l] for l in sorted_labels])
        pt_colors = [
            _hex_to_rgba(cmap.get(label_to_cluster.get(l, -1), "#FFF"), 0.9)
            for l in sorted_labels
        ]
        return edges_arr, edge_color, points, pt_colors

    viewer = napari.Viewer()

    # --- Scale bar in physical units -----------------------------------
    # World space is um because (a) image/label layers carry scale=VOXEL_SIZE
    # and (b) point/vector coords are already in um with scale=(1,1,1).
    viewer.scale_bar.visible = True
    viewer.scale_bar.unit = SCALE_BAR_UNIT
    viewer.scale_bar.colored = False
    # viewer.scale_bar.length = 20  # uncomment to pin the bar to exactly 20 um

    # --- Source fluorescence stack (original microscopy data) ---
    if PLOT_OPTIONS["source_fluorescence"]:
        _src_path = stack_records.get(TARGET_STACK, {}).get("source_stack_path", "")
        if _src_path and not os.path.isabs(_src_path):
            _src_path = os.path.join("workspace", _src_path)
        if _src_path and os.path.exists(_src_path):
            try:
                _raw = tifffile.imread(_src_path)
                # Detect channel axis (small dim <= 10, smaller than neighbor)
                if _raw.ndim == 3:
                    # Grayscale (Z, Y, X) -> single layer
                    viewer.add_image(
                        _raw, name="source ch0",
                        colormap="inferno", blending="additive",
                        scale=SOURCE_VOXEL,
                        opacity=0.7, visible=False,
                    )
                elif _raw.ndim == 4:
                    # Detect channel axis: smallest leading dim that's <= 10
                    if _raw.shape[0] <= 10 and _raw.shape[0] < _raw.shape[1]:
                        ch_axis = 0  # CZYX
                    elif _raw.shape[1] <= 10 and _raw.shape[1] < _raw.shape[0]:
                        ch_axis = 1  # ZCYX
                    else:
                        ch_axis = 0  # default fallback
                    n_ch = _raw.shape[ch_axis]
                    _cmaps = ["magenta", "green", "cyan", "yellow", "red", "blue"]
                    for _ci in range(n_ch):
                        if ch_axis == 0:
                            _vol = _raw[_ci]
                        else:
                            _vol = _raw[:, _ci]
                        viewer.add_image(
                            _vol, name=f"source ch{_ci}",
                            colormap=_cmaps[_ci % len(_cmaps)],
                            blending="additive",
                            scale=SOURCE_VOXEL,
                            opacity=0.7, visible=False,
                        )
                    print(f"  Source fluorescence: {n_ch} channels added (hidden by default)")
                else:
                    print(f"  [WARN] source stack has unexpected shape {_raw.shape}; skipped")
                # sanity: source and labels should share a physical extent
                _lab_extent = np.array(labels.shape) * np.array(VOXEL_SIZE)
                _src_shape = _raw.shape[-3:] if _raw.ndim >= 3 else _raw.shape
                _src_extent = np.array(_src_shape) * np.array(SOURCE_VOXEL)
                if not np.allclose(_lab_extent, _src_extent, rtol=0.02):
                    print(f"  [WARN] extent mismatch: labels {_lab_extent.round(1)} um "
                          f"vs source {_src_extent.round(1)} um -> check SOURCE_VOXEL")
                del _raw
            except Exception as _e:
                print(f"  [WARN] failed to load source stack: {_e}")
        else:
            print(f"  [WARN] source_fluorescence requested but no source_stack_path at: {_src_path or '<missing in state.json>'}")

    # --- Raw segmentation labels (each cell = unique napari default color) ---
    if PLOT_OPTIONS["raw_segmentation"]:
        viewer.add_labels(
            labels, name="segmentation",
            scale=VOXEL_SIZE,
            opacity=0.6, visible=False,  # hidden by default; busy when on
        )

    # --- Shape cluster layer ---
    shape_col = f"leiden_r{SHAPE_RESOLUTION}"
    if PLOT_OPTIONS["shape_clusters"] and shape_col in stack_df.columns:
        vol, dcmap = _cluster_volume_and_cmap(shape_col, SHAPE_CLUSTER_CMAP)
        viewer.add_labels(
            vol, name=f"shape r{SHAPE_RESOLUTION}",
            colormap=dcmap, scale=VOXEL_SIZE,
            opacity=0.7, visible=True,
        )

    # --- Niche cluster layers (one per config) ---
    niche_configs_for_plot = {}  # cache per-config cluster col + cmap for adjacency below
    if PLOT_OPTIONS["niche_configs"]:
        _selected = (
            list(niche_results_dict.keys())
            if NICHE_CONFIGS_TO_PLOT is None
            else [n for n in NICHE_CONFIGS_TO_PLOT if n in niche_results_dict]
        )
        for nc_name in _selected:
            nrdata = niche_results_dict[nc_name]
            vmask = nrdata["valid_mask"]
            membership = np.full(len(df), np.nan, dtype=float)
            membership[vmask] = nrdata["clusters"][NICHE_RESOLUTION]["membership"].astype(float)
            stack_idx = stack_df.index.values
            col_name = f"_nc_{nc_name}"
            stack_df[col_name] = membership[stack_idx]

            # Use the canonical (config, resolution)-keyed cmap so cluster N gets the
            # same color in the napari layer, the gallery, the adjacency
            # edges, and the exploded view -- even on stacks that are
            # missing some cluster IDs.
            nc_cmap = get_niche_cmap(nc_name, NICHE_RESOLUTION)
            vol, dcmap = _cluster_volume_and_cmap(col_name, nc_cmap)
            viewer.add_labels(
                vol, name=f"niche {nc_name} r{NICHE_RESOLUTION}",
                colormap=dcmap, scale=VOXEL_SIZE, opacity=0.7,
                visible=(nc_name == NICHE_CONFIG),
            )
            niche_configs_for_plot[nc_name] = (col_name, nc_cmap)

    # --- Orientation vectors (per selected niche config, per-cluster) ---
    # Centroids are already in um; vectors are unit axes scaled to VECTOR_LEN um.
    orient_cols = ["major_axis_z", "major_axis_y", "major_axis_x"]
    cent_cols = ["centroid-0", "centroid-1", "centroid-2"]
    niche_col = f"niche_leiden_r{NICHE_RESOLUTION}"
    if (PLOT_OPTIONS["orientation_vectors"]
            and all(c in stack_df.columns for c in cent_cols + orient_cols)):
        for _cfg_name, (_col_name, _cfg_cmap) in niche_configs_for_plot.items():
            _vdf = stack_df.dropna(subset=[_col_name])
            _clusters = sorted(_vdf[_col_name].astype(int).unique())
            for cid in _clusters:
                sub = _vdf[_vdf[_col_name].astype(int) == cid]
                if sub.empty:
                    continue
                centroids = sub[cent_cols].values          # um
                dirs = sub[orient_cols].values * VECTOR_LEN  # um
                # canonical +Z sign flip (major_axis has arbitrary +/- sign)
                sign = np.sign(dirs[:, 0])
                sign[sign == 0] = 1
                dirs = dirs * sign[:, None]
                # Optionally recenter so the centroid sits at the vector midpoint
                starts = centroids - dirs / 2 if CENTROID_AS_MIDPOINT else centroids
                vectors = np.stack([starts, dirs], axis=1)
                viewer.add_vectors(
                    vectors, name=f"NC{cid} orient ({_cfg_name})",
                    edge_color=_cfg_cmap.get(int(cid), "#888"),
                    edge_width=VECTOR_WIDTH, length=1,
                    vector_style=VECTOR_STYLE,  # default "line" (no arrowhead)
                    visible=(_cfg_name == NICHE_CONFIG),  # featured config shown
                )

    # --- Adjacency + centroids (reads <stack>_adjacency.csv if present) ---
    adj_path = ""
    record = stack_records.get(TARGET_STACK, {})
    fcsv = record.get("qc_features_csv", "")
    if fcsv:
        if not os.path.isabs(fcsv):                 # recorded relative to the workspace root
            fcsv = os.path.join("workspace", fcsv)
        adj_path = os.path.join(os.path.dirname(fcsv), TARGET_STACK + "_adjacency.csv")
    have_adj = bool(adj_path) and os.path.exists(adj_path)

    if PLOT_OPTIONS["adjacency_edges"] and have_adj:
        # Shape adjacency (default visible) + niche adjacency (featured config)
        if shape_col in stack_df.columns:
            result = _build_adjacency(shape_col, get_shape_cmap(SHAPE_RESOLUTION), adj_path)
            if result is not None:
                edges_arr, edge_color, _points, _pt_colors = result
                viewer.add_vectors(
                    edges_arr, name="adjacency [shape]",
                    edge_color=edge_color, edge_width=EDGE_WIDTH, length=1,
                    vector_style="line", opacity=0.6, visible=True,
                )
                print(f"  Shape adjacency: {len(edges_arr)} edges")
        if niche_col in stack_df.columns:
            result = _build_adjacency(niche_col, get_niche_cmap(NICHE_CONFIG, NICHE_RESOLUTION), adj_path)
            if result is not None:
                edges_n, ecolor_n, _pts, _ptc = result
                viewer.add_vectors(
                    edges_n, name=f"adjacency [niche {NICHE_CONFIG}]",
                    edge_color=ecolor_n, edge_width=EDGE_WIDTH, length=1,
                    vector_style="line", opacity=0.6, visible=False,
                )
                print(f"  Niche adjacency: {len(edges_n)} edges")
    elif PLOT_OPTIONS["adjacency_edges"] and not have_adj:
        print(f"  [WARN] adjacency_edges requested but no adjacency CSV at: {adj_path or '<no qc_features_csv>'}")

    # --- Cell centroids (one layer per coloring) ---
    if PLOT_OPTIONS["cell_centroids"] and have_adj:
        # Reuse centroids computed inside _build_adjacency via a fresh call
        # (cheap — just the centroid lookup + color mapping)
        if shape_col in stack_df.columns:
            result = _build_adjacency(shape_col, get_shape_cmap(SHAPE_RESOLUTION), adj_path)
            if result is not None:
                _edges, _ec, points, pt_colors = result
                viewer.add_points(
                    points, name="centroids [shape]",
                    face_color=pt_colors, size=CENTROID_SIZE, symbol="disc",
                    visible=True,
                )
        if niche_col in stack_df.columns:
            result = _build_adjacency(niche_col, get_niche_cmap(NICHE_CONFIG, NICHE_RESOLUTION), adj_path)
            if result is not None:
                _edges, _ec, pts, ptc = result
                viewer.add_points(
                    pts, name=f"centroids [niche {NICHE_CONFIG}]",
                    face_color=ptc, size=CENTROID_SIZE, symbol="disc",
                    visible=False,
                )

    _extent_um = np.array(labels.shape) * np.array(VOXEL_SIZE)
    print(f"Launched napari on: {TARGET_STACK[:60]}")
    print(f"Featured niche config: {NICHE_CONFIG}")
    print(f"World units: {SCALE_BAR_UNIT}  |  voxel {VOXEL_SIZE} um  |  "
          f"volume {_extent_um[0]:.1f} x {_extent_um[1]:.1f} x {_extent_um[2]:.1f} um (z,y,x)")
    print(f"Layers enabled: " + ", ".join(k for k, v in PLOT_OPTIONS.items() if v))
    print("Toggle layers in the napari layer list to A/B compare.")
    # NOTE: do NOT `del labels` here — downstream cells (exploded view, etc)
    # reference it. napari already holds its own reference for the layers
    # so the var doesn't actually free memory until the kernel exits.

In [ ]:
viewer.scale_bar.font_size = 60
viewer.scale_bar.length = 50
viewer.scale_bar.box = True
viewer.scale_bar.box_color = (0,0,0,0.75)

# set the bounding boxes:
for item in viewer.layers:
    if item.name == 'adjacency [niche hop3_pure]':
        item.bounding_box.visible = False
    else:
        item.bounding_box.visible = True
        item.bounding_box.line_color = (1,1,1,0.25)
        item.bounding_box.point_color = (1,1,1,0.25)
        item.bounding_box.line_thickness = 5

# save and export

In [ ]:
viewer.camera.angles = (33.64283414358399, -25.826384821218156, 124.97422150761433) #unpatterned skin

In [ ]:
viewer.export_figure(r"figures/Fig6H_isometric_unpatterned_adjacency.tif", scale_factor=2)

## Exploded cluster view (3D)

Builds a translated copy of the labels volume where each cluster is shifted
in 3D and cells are pushed radially from their cluster centroid. Useful for:
- inspecting a single cluster's spatial coherence
- figure-quality 3D views that 3D segmentation papers rarely show

**Controls.** Two layers of granularity:
- *Global defaults* (`EXPLODED_Z_SEPARATION`, `EXPLODED_RADIAL_FACTOR`) apply
  to any cluster you don't explicitly configure.
- *Per-cluster overrides* (`EXPLODED_PER_CLUSTER`) let you set a custom
  `(z, y, x)` center offset and radial push for each cluster individually —
  useful for manually arranging clusters into a clean publication layout.

Centroids come from the precomputed feature table (`centroid-0/1/2` in
`stack_df`). The only thing computed here is the cluster aggregate centroid
(mean of member cell centroids).

Run AFTER the napari launcher — depends on `viewer`, `labels`, `stack_df`,
`_hex_to_rgba`, `niche_col`, `shape_col`, etc. from its scope.


In [ ]:
# =======================================================================
# EXPLODED CLUSTER VIEW — separate clusters in 3D + push cells radially
# =======================================================================
# For each cell of cluster c:
#   new_centroid = cluster_centroid
#                + (cell_centroid - cluster_centroid) * radial_factor
#                + cluster_center_offset  (a full (z, y, x) vector)
#
# Centroids: pulled directly from precomputed `centroid-0/1/2` columns in
# stack_df (no recomputation). Cluster centroid = mean of member centroids.

# ---- USER KNOBS ----
EXPLODED_CLUSTER_COL    = niche_col       # column in stack_df: niche_col, shape_col, or "_nc_<config>"
EXPLODED_TARGET_CLUSTERS = None           # None = all clusters; or list like [0, 2, 5]

# Global defaults (apply to any cluster NOT in EXPLODED_PER_CLUSTER)
EXPLODED_Z_SEPARATION   = 100              # default Z offset = cluster_index * this value
EXPLODED_RADIAL_FACTOR  = 1.5             # 1.0 = original positions; 2.0 = doubled distance from cluster center

# Per-cluster overrides — cluster_id -> dict with optional keys:
#   "center_offset": (z, y, x) tuple of voxel offsets for this cluster's center
#                    Overrides the auto Z separation. Use (0, 0, 0) to leave at original position.
#   "radial":        float — overrides EXPLODED_RADIAL_FACTOR for this cluster
#
# Example — manually arrange three clusters into a "trifold" layout:
#   EXPLODED_PER_CLUSTER = {
#       0: {"center_offset": (   0,    0,    0), "radial": 1.0},
#       1: {"center_offset": ( 200,    0,  300), "radial": 2.5},
#       2: {"center_offset": ( 400,  300,  -50)},  # uses default radial
#   }
EXPLODED_PER_CLUSTER    = None             # None = use global defaults for all

EXPLODED_OUTPUT_PADDING = 30
EXPLODED_LAYER_VISIBLE  = True

# Coloring of the OVERVIEW layer(s) (whole exploded volume in one layer):
#   "raw_label" — each cell its own napari-default color (default)
#   "cluster"   — every cell of a cluster gets the same cluster color
#   "both"      — adds both overview layers
EXPLODED_COLOR_MODE     = "raw_label"

# Per-cluster layers — one labels layer per cluster.
# Each is visible by default; toggle individually in the napari layer list.
EXPLODED_PER_CLUSTER_LAYERS     = True   # cluster-colored per-cluster layer (single color)
EXPLODED_PER_CLUSTER_RAW_LAYERS = False  # raw-label-colored per-cluster layer (each cell unique color)
                                         # default OFF — heavy memory; the raw_label overview already
                                         # shows every cell uniquely colored. Set True if you want
                                         # cells per-cluster individually distinct in their own layer.

# Where to put the exploded layers:
#   True  - open a NEW napari window for the exploded view (keeps main viewer clean,
#           gives independent camera state, ideal for manuscript screenshots)
#   False - add to the existing `viewer` (the one cell 82 created)
EXPLODED_USE_SEPARATE_VIEWER = True

# Cluster stacking order along Z. None (default) = sorted (0, 1, 2, ...).
# Provide an explicit list to control hierarchy, e.g. [1, 3, 2] places
# cluster 1 lowest, then cluster 3, then cluster 2 on top.
# Doubles as a filter: only clusters in the list are rendered (any in the
# list but not present in the data are silently skipped with a warning).
# EXPLODED_CLUSTER_ORDER = [2,6,8,0,3,1]
EXPLODED_CLUSTER_ORDER = None


# Visualization downsample for the exploded view ONLY (does not touch the
# main viewer). 1 = full resolution. 2 = 8x less memory + ~5x faster render
# (cell shapes still clear). 4 = 64x less memory; coarse but fine for figure
# overview. Useful when the volume is multi-billion voxels.
EXPLODED_DOWNSAMPLE = 2
EXPLODED_MAX_GB         = 50.0              # safety: refuse outputs larger than this
# --------------------


def _per_cluster_params(cid, c_idx, per_cluster, default_z_sep, default_radial):
    """Resolve per-cluster center_offset + radial, falling back to globals."""
    over = (per_cluster or {}).get(cid, {}) if per_cluster else {}
    center_offset = over.get("center_offset", (c_idx * default_z_sep, 0.0, 0.0))
    radial = over.get("radial", default_radial)
    return np.asarray(center_offset, dtype=float), float(radial)


def build_exploded_volume(orig_labels, sdf, cluster_col,
                          target_clusters=None,
                          z_separation=0, radial_factor=1.0,
                          per_cluster=None,
                          padding=0, max_gb=8.0,
                          downsample=1,
                          cluster_order=None):
    """Return (new_volume, new_centroids_dict) or (None, {}) on abort/no-data."""
    from scipy.ndimage import find_objects

    cent_cols = ["centroid-0", "centroid-1", "centroid-2"]   # precomputed in feature table
    valid = sdf.dropna(subset=[cluster_col, "label"] + cent_cols).copy()
    valid[cluster_col] = valid[cluster_col].astype(int)
    valid["label"] = valid["label"].astype(int)

    # CENTROIDS ARE IN PHYSICAL UNITS (microns), NOT VOXELS.
    # Divide by SPACING to convert to voxel coords (and by downsample factor
    # if we slice the labels volume to lower resolution for visualization).
    try:
        _spacing = float(SPACING)
    except NameError:
        _spacing = 1.0
        print("  [WARN] SPACING not defined — assuming centroids are already voxel coords")
    _scale = _spacing * max(int(downsample), 1)
    if _scale != 1.0:
        for _c in cent_cols:
            valid[_c] = valid[_c] / _scale

    # Downsample input labels for visualization (per-axis stride). Cells
    # smaller than `downsample` voxels in any dim may disappear; for typical
    # cell sizes (>5 voxels) this is fine for figure-quality rendering.
    if downsample > 1:
        orig_labels = orig_labels[::downsample, ::downsample, ::downsample].copy()
        print(f"  Downsampled labels by {downsample}x -> {orig_labels.shape}")
    if target_clusters is not None:
        valid = valid[valid[cluster_col].isin(target_clusters)]
    if valid.empty:
        print(f"  No cells match filter (cluster_col={cluster_col}, target={target_clusters})")
        return None, {}

    _all_present = sorted(valid[cluster_col].unique())
    if cluster_order is not None:
        _missing = [c for c in cluster_order if c not in _all_present]
        if _missing:
            print(f"  [WARN] cluster_order has IDs not in data: {_missing}")
        cluster_ids = [c for c in cluster_order if c in _all_present]
    else:
        cluster_ids = _all_present
    print(f"  Clusters (Z order, bottom -> top): {cluster_ids}")

    # New centroid per cell
    new_centroids = {}
    for c_idx, cid in enumerate(cluster_ids):
        sub = valid[valid[cluster_col] == cid]
        c_cent = sub[cent_cols].mean().values         # cluster aggregate centroid
        center_offset, radial = _per_cluster_params(
            int(cid), c_idx, per_cluster, z_separation, radial_factor,
        )
        for _, row in sub.iterrows():
            lid = int(row["label"])
            cell_cent = row[cent_cols].values.astype(float)
            offset = cell_cent - c_cent
            ncent = c_cent + offset * radial + center_offset
            new_centroids[lid] = ncent

    # Compute output bbox by translating each cell's original bbox
    objects = find_objects(orig_labels)
    new_bboxes = {}
    z_bounds = [0, 0]; y_bounds = [0, 0]; x_bounds = [0, 0]
    for lid, ncent in new_centroids.items():
        if lid - 1 >= len(objects) or objects[lid - 1] is None:
            continue
        zs, ys, xs = objects[lid - 1]
        obox_center = np.array([(zs.start + zs.stop) / 2,
                                (ys.start + ys.stop) / 2,
                                (xs.start + xs.stop) / 2])
        delta = np.round(ncent - obox_center).astype(int)
        nz = slice(zs.start + int(delta[0]), zs.stop + int(delta[0]))
        ny = slice(ys.start + int(delta[1]), ys.stop + int(delta[1]))
        nx = slice(xs.start + int(delta[2]), xs.stop + int(delta[2]))
        new_bboxes[lid] = (nz, ny, nx, zs, ys, xs)
        z_bounds[0] = min(z_bounds[0], nz.start); z_bounds[1] = max(z_bounds[1], nz.stop)
        y_bounds[0] = min(y_bounds[0], ny.start); y_bounds[1] = max(y_bounds[1], ny.stop)
        x_bounds[0] = min(x_bounds[0], nx.start); x_bounds[1] = max(x_bounds[1], nx.stop)

    z_size = (z_bounds[1] - z_bounds[0]) + 2 * padding
    y_size = (y_bounds[1] - y_bounds[0]) + 2 * padding
    x_size = (x_bounds[1] - x_bounds[0]) + 2 * padding
    z_shift = -z_bounds[0] + padding
    y_shift = -y_bounds[0] + padding
    x_shift = -x_bounds[0] + padding

    # Downcast to smallest fitting integer dtype to save memory
    _max_lid = int(orig_labels.max())
    if _max_lid < 256:
        _out_dtype = np.uint8
    elif _max_lid < 65536:
        _out_dtype = np.uint16
    else:
        _out_dtype = np.uint32

    mem_gb = (z_size * y_size * x_size * np.dtype(_out_dtype).itemsize) / 1e9
    print(f"  Exploded volume shape: ({z_size}, {y_size}, {x_size}) ~{mem_gb:.2f} GB")
    if mem_gb > max_gb:
        print(f"  ABORT: would exceed EXPLODED_MAX_GB={max_gb} GB. "
              f"Reduce factors or set EXPLODED_TARGET_CLUSTERS to a subset.")
        return None, {}

    out_vol = np.zeros((z_size, y_size, x_size), dtype=_out_dtype)
    for lid, (nz, ny, nx, ozs, oys, oxs) in new_bboxes.items():
        cell_mask = (orig_labels[ozs, oys, oxs] == lid)
        nz2 = slice(nz.start + z_shift, nz.stop + z_shift)
        ny2 = slice(ny.start + y_shift, ny.stop + y_shift)
        nx2 = slice(nx.start + x_shift, nx.stop + x_shift)
        out_vol[nz2, ny2, nx2][cell_mask] = lid
    return out_vol, new_centroids


# Defensive: reload `labels` from disk if it was freed/never set
# (e.g. if cell 82 was edited or kernel restarted)
if "labels" not in dir() or labels is None:
    import tifffile as _tf
    labels = _tf.imread(stack_paths[TARGET_STACK])
    print(f"  Reloaded labels from disk: {labels.shape}")

def _resolve_cluster_cmap(cluster_col):
    """Return the canonical cluster cmap for any cluster column.
    Routes through the (config, resolution)-keyed registry
    (get_shape_cmap / get_niche_cmap) so the exploded view matches
    the napari labels, the adjacency edges, and the gallery for the
    same cluster IDs.
    """
    try:
        if cluster_col == niche_col:
            return get_niche_cmap(NICHE_CONFIG, NICHE_RESOLUTION)
        if cluster_col == shape_col:
            return get_shape_cmap(SHAPE_RESOLUTION)
        if cluster_col.startswith("_nc_"):
            _name = cluster_col[4:]
            return get_niche_cmap(_name, NICHE_RESOLUTION)
    except NameError:
        pass
    # Fallback for unrecognised columns: build from values present in stack_df.
    _vals = sorted(stack_df[cluster_col].dropna().astype(int).unique())
    return build_cluster_cmap(_vals, base=_NICHE_BASE)


# Build it
print(f"Building exploded view: cluster_col={EXPLODED_CLUSTER_COL!r} "
      f"radial={EXPLODED_RADIAL_FACTOR}x z_sep={EXPLODED_Z_SEPARATION}")
_exp_vol, _exp_centroids = build_exploded_volume(
    labels, stack_df, EXPLODED_CLUSTER_COL,
    target_clusters=EXPLODED_TARGET_CLUSTERS,
    z_separation=EXPLODED_Z_SEPARATION,
    radial_factor=EXPLODED_RADIAL_FACTOR,
    per_cluster=EXPLODED_PER_CLUSTER,
    padding=EXPLODED_OUTPUT_PADDING,
    max_gb=EXPLODED_MAX_GB,
    downsample=EXPLODED_DOWNSAMPLE,
    cluster_order=EXPLODED_CLUSTER_ORDER,
)

if _exp_vol is not None:
    # Choose target napari viewer
    if EXPLODED_USE_SEPARATE_VIEWER:
        import napari as _napari
        _exp_viewer = _napari.Viewer(
            title=f"Exploded — {EXPLODED_CLUSTER_COL} (r={EXPLODED_RADIAL_FACTOR}x, z={EXPLODED_Z_SEPARATION})",
            ndisplay=3,
        )
        print(f"  Opened new napari window for exploded view")
    else:
        _exp_viewer = viewer

    _modes = ["raw_label", "cluster"] if EXPLODED_COLOR_MODE == "both" else [EXPLODED_COLOR_MODE]

    for _mode in _modes:
        if _mode == "raw_label":
            # Each cell -> its own napari-default color (cells visually distinct)
            _layer_name = (
                f"exploded {EXPLODED_CLUSTER_COL} [raw] "
                f"(r={EXPLODED_RADIAL_FACTOR}x, z={EXPLODED_Z_SEPARATION})"
            )
            _exp_viewer.add_labels(
                _exp_vol, name=_layer_name,
                opacity=0.85,
                visible=EXPLODED_LAYER_VISIBLE,
            )
            print(f"  Added '{_layer_name}' with {len(_exp_centroids)} cells (raw label coloring)")

        elif _mode == "cluster":
            _exp_cmap_dict = _resolve_cluster_cmap(EXPLODED_CLUSTER_COL)

            _exp_max_label = int(_exp_vol.max())
            _lut = np.zeros(_exp_max_label + 1, dtype=np.int32)
            _vc = stack_df.dropna(subset=[EXPLODED_CLUSTER_COL])
            for _, _row in _vc.iterrows():
                _lid = int(_row["label"])
                _cid = int(_row[EXPLODED_CLUSTER_COL])
                if 0 < _lid <= _exp_max_label:
                    _lut[_lid] = _cid + 1
            _exp_cluster_vol = _lut[_exp_vol]

            _color_dict = {None: (0, 0, 0, 0), 0: (0, 0, 0, 0)}
            for _cid in sorted(_vc[EXPLODED_CLUSTER_COL].astype(int).unique()):
                _color_dict[int(_cid) + 1] = _hex_to_rgba(_exp_cmap_dict.get(int(_cid), "#888"))
            _exp_dcmap = DirectLabelColormap(color_dict=_color_dict)

            _layer_name = (
                f"exploded {EXPLODED_CLUSTER_COL} [cluster] "
                f"(r={EXPLODED_RADIAL_FACTOR}x, z={EXPLODED_Z_SEPARATION})"
            )
            _exp_viewer.add_labels(
                _exp_cluster_vol, name=_layer_name,
                colormap=_exp_dcmap, opacity=0.85,
                visible=EXPLODED_LAYER_VISIBLE,
            )
            print(f"  Added '{_layer_name}' with {len(_exp_centroids)} cells (cluster coloring)")
            del _exp_cluster_vol

        else:
            print(f"  [WARN] Unknown EXPLODED_COLOR_MODE: {_mode!r}")

    # --- Per-cluster layers (one per cluster, cluster-colored) ---
    if EXPLODED_PER_CLUSTER_LAYERS or EXPLODED_PER_CLUSTER_RAW_LAYERS:
        _pc_cmap_dict = _resolve_cluster_cmap(EXPLODED_CLUSTER_COL)

        # Map each label -> its cluster id (or None for cells we don't expose)
        _pc_label_to_cluster = {}
        _pc_valid = stack_df.dropna(subset=[EXPLODED_CLUSTER_COL])
        for _, _row in _pc_valid.iterrows():
            _pc_label_to_cluster[int(_row["label"])] = int(_row[EXPLODED_CLUSTER_COL])

        _pc_clusters = sorted(set(_pc_label_to_cluster.values()))
        if EXPLODED_TARGET_CLUSTERS is not None:
            _pc_clusters = [c for c in _pc_clusters if c in EXPLODED_TARGET_CLUSTERS]

        for _pc_cid in _pc_clusters:
            # Mask: keep only labels that belong to this cluster
            _pc_mask = np.zeros(int(_exp_vol.max()) + 1, dtype=bool)
            for _lid, _cid in _pc_label_to_cluster.items():
                if _cid == _pc_cid and 0 < _lid < _pc_mask.shape[0]:
                    _pc_mask[_lid] = True
            _pc_lookup = _pc_mask[_exp_vol]   # bool: True where this cluster's cells live

            if EXPLODED_PER_CLUSTER_LAYERS:
                # Cluster-colored layer: collapse to a single label value (cid+1)
                # so napari only sees 2 unique values (0 + cid+1) -> fast render,
                # uniform hover label.
                # uint8 fits up to 255 clusters; collapse to single value
                _pc_collapsed = np.where(_pc_lookup,
                                          np.uint8(_pc_cid + 1), np.uint8(0))
                _pc_color = _hex_to_rgba(_pc_cmap_dict.get(int(_pc_cid), "#888"))
                _pc_dcmap = DirectLabelColormap(color_dict={
                    None: (0, 0, 0, 0),
                    0: (0, 0, 0, 0),
                    int(_pc_cid + 1): _pc_color,
                })
                _pc_name_layer = f"exploded NC{_pc_cid} ({EXPLODED_CLUSTER_COL})"
                _exp_viewer.add_labels(
                    _pc_collapsed, name=_pc_name_layer,
                    colormap=_pc_dcmap, opacity=0.85,
                    visible=False,  # off by default — toggle on in napari to focus on this cluster
                )
                del _pc_collapsed

            if EXPLODED_PER_CLUSTER_RAW_LAYERS:
                # Raw-label layer: keep ORIGINAL label IDs so each cell gets its
                # own napari-default color. (No colormap passed -> auto.)
                _pc_raw_data = np.where(_pc_lookup, _exp_vol, 0)
                _pc_raw_name = f"exploded NC{_pc_cid} [raw] ({EXPLODED_CLUSTER_COL})"
                _exp_viewer.add_labels(
                    _pc_raw_data, name=_pc_raw_name,
                    opacity=0.85,
                    visible=False,  # off by default — toggle on in napari to focus on this cluster
                )
                del _pc_raw_data
        _modes_done = []
        if EXPLODED_PER_CLUSTER_LAYERS: _modes_done.append("cluster-colored")
        if EXPLODED_PER_CLUSTER_RAW_LAYERS: _modes_done.append("raw-label")
        print(f"  Added {len(_pc_clusters)} per-cluster layers x {len(_modes_done)} mode(s) "
              f"({', '.join(_modes_done)})")

    del _exp_vol  # free memory
